# Lyapunov Spectrum Analysis — Sweep Group

Compute empirical and predicted Lyapunov spectra for every run in a W&B sweep group.

**Workflow:**
1. Select W&B project / group
2. For each run: load best-validation model, compute Lyapunov spectra on test trajectories
3. Optionally compute observed-space Jacobians via $J_E^{-1} J_\text{latent} J_E$
4. Cache results per-run, generate HTML reports

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import base64
import json
import time
import traceback
from datetime import datetime
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf
from tqdm.notebook import tqdm

from JacobianODE.jacobians.checkpoints.loader import load_run
from JacobianODE.jacobians.metrics import r2_score
from JacobianODE.jacobians.run_analytics import plot_lyapunov_spectrum
from JacobianODE.jacobians.tuning.sweep import discover_sweep_runs
from JacobianODE.models.latent_jacobian import LitLatentJacobianODE

torch.set_float32_matmul_precision("high")

## Configuration

In [13]:
WANDB_PROJECT = "WMTask_INDall_N1_D1_NormTrue_T128__JacobianODE"
# WANDB_GROUP = None
# WANDB_GROUP = "spline_coupling__sweep_lc_x_kl_dyn_vae_sample_all_losses"
WANDB_GROUP = "spline_coupling__sweep_lc_x_kl_dyn_30step_cleantarget"

ENTITY = "JacobianODE"
PROJECT = WANDB_PROJECT
GROUP = WANDB_GROUP or None
SAVE_DIR = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs"

# True Lyapunov exponents (comma-separated string, or empty)
# true_lyap_str = "0.91, 0.0, -14.57"
true_lyap_str = ""
TRUE_LYAPUNOV = (
    [float(x.strip()) for x in true_lyap_str.split(",") if x.strip()]
    if true_lyap_str
    else None
)

COMPUTE_OBS_JAC = True
FORCE_RECOMPUTE = False

PROJECT_PATH = f"{ENTITY}/{PROJECT}"

print(
    f"Config: {PROJECT_PATH} | group={GROUP} | obs_jac={COMPUTE_OBS_JAC} | "
    f"true_lyap={TRUE_LYAPUNOV}"
)

Config: JacobianODE/WMTask_INDall_N1_D1_NormTrue_T128__JacobianODE | group=spline_coupling__sweep_lc_x_kl_dyn_30step_cleantarget | obs_jac=True | true_lyap=None


In [14]:
discovered = discover_sweep_runs(
    ENTITY, PROJECT, wandb_group=GROUP, verbose=True
)
print(
    f"Found {len(discovered.run_ids)} runs. "
    f"IDs: {', '.join(discovered.run_ids[:10])}"
    + ("..." if len(discovered.run_ids) > 10 else "")
)

KeyboardInterrupt: 

In [ ]:
# Fetch W&B validation metrics for each run
import wandb as _wandb

_api = _wandb.Api(timeout=90)
wandb_val_metrics = {}
for _run_id in tqdm(discovered.run_ids, desc="Fetching W&B metrics"):
    try:
        _run = _api.run(f"{ENTITY}/{PROJECT}/{_run_id}")
        _summary = _run.summary
        wandb_val_metrics[_run_id] = {
            "trajectory_val_loss": _summary.get("trajectory val_loss"),
            "val_loop_closure_loss": _summary.get("val/loop_closure_loss"),
            "mean_val_loss": _summary.get("mean val loss"),
        }
    except Exception as _e:
        wandb_val_metrics[_run_id] = {
            "trajectory_val_loss": None,
            "val_loop_closure_loss": None,
            "mean_val_loss": None,
        }

print(f"Fetched W&B validation metrics for {len(wandb_val_metrics)} runs.")

Fetching W&B metrics:   0%|          | 0/79 [00:00<?, ?it/s]

Fetched W&B validation metrics for 79 runs.


## Utility Functions

In [15]:
def z_dyn_slice(z, n_target_dims):
    if n_target_dims is not None:
        return z[..., :n_target_dims]
    return z

def compute_encoder_jacobian(lit_model, traj_obs, n_target_dims, device):
    """Compute encoder Jacobian J_E at each point along a trajectory.

    Args:
        lit_model: trained LitLatentJacobianODE
        traj_obs: (T, D_obs) tensor in observation space
        n_target_dims: int or None
        device: torch device

    Returns:
        J_E: (T', D_dyn, D_obs) tensor
    """
    encoder = lit_model.encoder
    margin = getattr(encoder, "context_margin", 0)

    if hasattr(encoder, "time_window"):
        w = encoder.time_window
        D_raw = traj_obs.shape[-1]
        T_avail = traj_obs.shape[0] - w + 1
        windows = traj_obs.unfold(0, w, 1).permute(0, 2, 1)  # (T', w, D_raw)
        x_flat = windows.reshape(T_avail, w * D_raw)

        def encode_point(x_pt):
            z = encoder.encode(x_pt.reshape(1, w, D_raw)).squeeze(0)
            return z[:n_target_dims] if n_target_dims is not None else z
    else:
        T_prime = traj_obs.shape[0] - margin
        x_flat = traj_obs[margin : margin + T_prime]  # (T', D_obs)

        def encode_point(x_pt):
            z = encoder.encode(x_pt.unsqueeze(0).unsqueeze(0)).squeeze(0).squeeze(0)
            return z[:n_target_dims] if n_target_dims is not None else z

    D_obs = x_flat.shape[-1]
    D_dyn = n_target_dims if n_target_dims is not None else D_obs

    if D_dyn <= D_obs:
        jac_fn = torch.func.jacrev(encode_point)
    else:
        jac_fn = torch.func.jacfwd(encode_point)

    was_training = encoder.training
    encoder.eval()
    try:
        # Process in chunks to avoid OOM
        chunk_size = 64
        J_chunks = []
        for ci in range(0, x_flat.shape[0], chunk_size):
            chunk = x_flat[ci : ci + chunk_size]
            J_chunk = torch.func.vmap(jac_fn)(chunk)  # (chunk, D_dyn, D_obs)
            J_chunks.append(J_chunk)
        J_E = torch.cat(J_chunks, dim=0)
    finally:
        if was_training:
            encoder.train()

    return J_E  # (T', D_dyn, D_obs)

def compute_encoder_jacobian_batched(
    lit_model, traj_batch, n_target_dims, device, chunk_size=256
):
    """Batched encoder Jacobian: one vmap call for all (b, t) points together.

    Flattens the leading (B, T_prime) dims into one big batch, calls
    ``torch.func.vmap(jac_fn)`` in chunks of ``chunk_size`` flat points, and
    reshapes the result back to ``(B, T_prime, D_dyn, D_obs)``.

    Args:
        lit_model: trained LitLatentJacobianODE
        traj_batch: (B, T, D_raw) tensor in observation space, on ``device``
        n_target_dims: int or None
        device: torch device
        chunk_size: flat-point batch size per vmap call (not per traj)

    Returns:
        J_E: (B, T_prime, D_dyn, D_obs_flat) tensor, where for windowed
        encoders ``D_obs_flat = w * D_raw`` (matches the per-traj function).
    """
    encoder = lit_model.encoder
    margin = getattr(encoder, "context_margin", 0)
    B = traj_batch.shape[0]

    if hasattr(encoder, "time_window"):
        w = encoder.time_window
        D_raw = traj_batch.shape[-1]
        T_prime = traj_batch.shape[1] - w + 1
        # (B, T_prime, D_raw, w) -> (B, T_prime, w, D_raw)
        windows = traj_batch.unfold(1, w, 1).permute(0, 1, 3, 2)
        x_flat = windows.reshape(B * T_prime, w * D_raw)

        def encode_point(x_pt):
            z = encoder.encode(x_pt.reshape(1, w, D_raw)).squeeze(0)
            return z[:n_target_dims] if n_target_dims is not None else z
    else:
        T_prime = traj_batch.shape[1] - margin
        x_slice = traj_batch[:, margin : margin + T_prime]  # (B, T_prime, D_obs)
        x_flat = x_slice.reshape(B * T_prime, -1)

        def encode_point(x_pt):
            z = encoder.encode(x_pt.unsqueeze(0).unsqueeze(0)).squeeze(0).squeeze(0)
            return z[:n_target_dims] if n_target_dims is not None else z

    D_obs_flat = x_flat.shape[-1]
    D_dyn = n_target_dims if n_target_dims is not None else D_obs_flat

    if D_dyn <= D_obs_flat:
        jac_fn = torch.func.jacrev(encode_point)
    else:
        jac_fn = torch.func.jacfwd(encode_point)

    was_training = encoder.training
    encoder.eval()
    try:
        J_chunks = []
        total = x_flat.shape[0]
        for ci in range(0, total, chunk_size):
            chunk = x_flat[ci : ci + chunk_size]
            J_chunk = torch.func.vmap(jac_fn)(chunk)  # (chunk, D_dyn, D_obs_flat)
            J_chunks.append(J_chunk)
        J_E_flat = torch.cat(J_chunks, dim=0)  # (B*T_prime, D_dyn, D_obs_flat)
    finally:
        if was_training:
            encoder.train()

    return J_E_flat.reshape(B, T_prime, D_dyn, D_obs_flat)

def compute_observed_space_jacobians(lit_model, traj_single, n_target_dims, device):
    """Compute J_obs = J_E_pinv @ J_latent @ J_E for a single trajectory.

    Args:
        lit_model: trained LitLatentJacobianODE
        traj_single: (T, D_obs) tensor
        n_target_dims: int or None
        device: torch device

    Returns:
        J_obs: (T', D_obs, D_obs) tensor
    """
    traj = traj_single.unsqueeze(0).to(device)  # (1, T, D_obs)

    # 1. Encode -> z_dyn
    z_full = lit_model.encode_trajectory(traj)  # (1, T', D_latent)
    z_dyn = z_dyn_slice(z_full, n_target_dims)
    T_prime = z_dyn.shape[1]

    # 2. Latent Jacobians
    J_latent = lit_model.compute_jacobians(z_dyn)[0]  # (T', D_dyn, D_dyn)

    # 3. Encoder Jacobian
    # Align: J_E should have the same T' as J_latent
    J_E = compute_encoder_jacobian(
        lit_model, traj_single, n_target_dims, device
    )  # (T'', D_dyn, D_obs)
    # Trim to match (they should already match but be safe)
    T_min = min(J_E.shape[0], J_latent.shape[0])
    J_E = J_E[:T_min]
    J_latent = J_latent[:T_min]

    # 4. Pseudo-inverse
    J_E_pinv = torch.linalg.pinv(J_E)  # (T', D_obs, D_dyn)

    # 5. Assemble: J_obs = J_E_pinv @ J_latent @ J_E
    J_obs = J_E_pinv @ J_latent @ J_E  # (T', D_obs, D_obs)

    return J_obs

def compute_observed_space_jacobians_batched(
    lit_model, traj_batch, n_target_dims, device, chunk_size=256
):
    """Batched observed-space Jacobian across trajectories.

    Equivalent to calling ``compute_observed_space_jacobians`` per trajectory
    and stacking, but:
      * ``encode_trajectory``, ``compute_jacobians``, ``pinv`` and the matmul
        are all run once on the full batch.
      * The encoder-Jacobian vmap is flattened over (B, T_prime), so a single
        call covers points from many trajectories at once.

    Args:
        lit_model: trained LitLatentJacobianODE
        traj_batch: (B, T, D_obs) tensor, moved to ``device`` internally.
        n_target_dims: int or None
        device: torch device
        chunk_size: flat-point batch size for the encoder-Jacobian vmap.

    Returns:
        J_obs: (B, T', D_obs_flat, D_obs_flat) tensor.
    """
    traj_batch = traj_batch.to(device)

    # 1. Encode batch -> z_dyn
    z_full = lit_model.encode_trajectory(traj_batch)  # (B, T', D_latent)
    z_dyn = z_dyn_slice(z_full, n_target_dims)

    # 2. Latent Jacobians (already batched over leading dims)
    J_latent = lit_model.compute_jacobians(z_dyn)  # (B, T', D_dyn, D_dyn)

    # 3. Encoder Jacobian (batched over all (b, t) points)
    J_E = compute_encoder_jacobian_batched(
        lit_model, traj_batch, n_target_dims, device, chunk_size=chunk_size
    )  # (B, T_E, D_dyn, D_obs_flat)

    # Align T dims (should match, but be safe)
    T_min = min(J_E.shape[1], J_latent.shape[1])
    J_E = J_E[:, :T_min]
    J_latent = J_latent[:, :T_min]

    # 4. Pseudo-inverse (batched over B, T)
    J_E_pinv = torch.linalg.pinv(J_E)  # (B, T', D_obs_flat, D_dyn)

    # 5. Assemble: J_obs = J_E_pinv @ J_latent @ J_E
    J_obs = J_E_pinv @ J_latent @ J_E  # (B, T', D_obs_flat, D_obs_flat)

    return J_obs

def save_results(path, data):
    """Save results dict to JSON, converting numpy arrays to lists."""
    def convert_for_json(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.floating, np.integer)):
            return obj.item()
        if isinstance(obj, dict):
            return {k: convert_for_json(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [convert_for_json(v) for v in obj]
        return obj

    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(convert_for_json(data), f, indent=2)

def load_cached(path):
    """Load cached JSON results, or return None if missing/corrupt."""
    if path.exists():
        try:
            with open(path) as f:
                data = json.load(f)
            if data:
                return data
        except (json.JSONDecodeError, ValueError):
            path.unlink()  # remove corrupt cache file
    return None

def fig_to_base64(fig):
    """Convert matplotlib figure to base64-encoded PNG string."""
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return b64

def spectrum_comparison(pred_le, emp_le):
    """Compute correlation and R^2 between predicted and empirical LE spectra."""
    n = min(len(pred_le), len(emp_le))
    if n == 0:
        return float("nan"), float("nan")
    p = np.asarray(pred_le[:n], dtype=np.float64)
    e = np.asarray(emp_le[:n], dtype=np.float64)
    corr = float(np.corrcoef(p, e)[0, 1]) if n > 1 else float("nan")
    r2 = float(r2_score(torch.tensor(e), torch.tensor(p)))
    return corr, r2

def spectrum_mse(pred_le, emp_le):
    """MSE between predicted and empirical spectra (first min(len) exponents)."""
    n = min(len(pred_le), len(emp_le))
    if n == 0:
        return float("nan")
    p = np.asarray(pred_le[:n], dtype=np.float64)
    e = np.asarray(emp_le[:n], dtype=np.float64)
    return float(np.mean((p - e) ** 2))

def lyap_to_timescale(le_array):
    """Convert Lyapunov exponents to timescales: 1/|lambda|.

    Timescales de-emphasise large negative exponents (fast-decaying modes)
    and better reflect dynamically relevant time horizons.
    """
    le = np.asarray(le_array, dtype=np.float64)
    with np.errstate(divide="ignore"):
        ts = 1.0 / np.abs(le)
    ts[~np.isfinite(ts)] = np.nan  # lambda=0 -> nan timescale
    return ts

## Main Processing Loop

In [ ]:
# Main processing loop
_repo_root = Path.cwd().parent if Path.cwd().name == "_notebook" else Path.cwd()
results_dir = _repo_root / "_notebook" / "results" / "lyapunov" / PROJECT_PATH.split("/")[-1]
results_dir.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_results = {}

# Shared data (loaded once from first run)
_eq_shared = None
_trajs_shared = None
_dt_shared = None
_cfg_shared = None
_data_loaded = False

# Chunk size for batched Jacobian / Lyapunov compute (matches run_analytics.py)
_LYAP_CHUNK = 64
# Flat-point chunk for the batched encoder-Jacobian vmap.
_ENC_JAC_CHUNK = 256
# Trajectory chunk size for the batched observed-space Jacobian path.
_OBS_JAC_TRAJ_CHUNK = 32

for _ri, _run_id in enumerate(tqdm(discovered.run_ids, desc="Processing runs")):
    # Check cache
    _cache_path = results_dir / f"{_run_id}.json"
    if not FORCE_RECOMPUTE:
        _cached = load_cached(_cache_path)
        if _cached is not None:
            all_results[_run_id] = _cached
            print(f"  [{_ri+1}/{len(discovered.run_ids)}] {_run_id}: loaded from cache")
            continue

    print(f"  [{_ri+1}/{len(discovered.run_ids)}] {_run_id}: recomputing ...")
    try:
        # Load model
        print(f"    [load] loading run + model ...")
        (
            _run_obj,
            _cfg,
            _eq,
            _dt,
            _values,
            _train_dl,
            _val_dl,
            _test_dl,
            _trajs,
            _lit_model,
        ) = load_run(
            PROJECT_PATH,
            run_id=_run_id,
            save_dir=SAVE_DIR,
            generate_data=(not _data_loaded),
            verbose=False,
        )

        if not _data_loaded:
            _eq_shared = _eq
            _trajs_shared = _trajs
            _dt_shared = _dt
            _cfg_shared = _cfg
            _data_loaded = True

        # Ensure best checkpoint is loaded
        from JacobianODE.jacobians.checkpoints.loader import load_checkpoint
        try:
            load_checkpoint(
                _run_obj, _cfg, _lit_model,
                save_dir=SAVE_DIR, verbose=False,
            )
        except Exception:
            pass  # checkpoint already loaded by load_run

        _lit_model.eval()
        _lit_model = _lit_model.to(device)

        # Extract config
        _n_target_dims = OmegaConf.select(_cfg, "model.n_target_dims", default=None)
        if _n_target_dims is not None:
            _n_target_dims = int(_n_target_dims)
        _lc_weight = float(OmegaConf.select(_cfg, "training.lightning.loop_closure_weight", default=0.0))
        _kl_dyn = float(OmegaConf.select(_cfg, "training.lightning.kl_dyn_weight", default=0.0))

        # Normalization params
        _mu = np.asarray(OmegaConf.select(_cfg_shared, "data.postprocessing.mu", default=0.0))
        _sigma = np.asarray(OmegaConf.select(_cfg_shared, "data.postprocessing.sigma", default=1.0))

        # Test trajectories
        if "test_trajs_full" in _trajs_shared:
            _test_seq = _trajs_shared["test_trajs_full"].sequence
        else:
            _test_seq = _trajs_shared["test_trajs"].sequence
        _n_test = _test_seq.shape[0]

        # Delay embedding info
        _delay_params = OmegaConf.select(_cfg_shared, "data.train_test_params.delay_embedding_params", default=None)
        _n_delays = int(_delay_params.get("n_delays", 1)) if _delay_params else 1

        print(
            f"    [config] n_test={_n_test}, T={_test_seq.shape[1]}, D_obs={_test_seq.shape[-1]}, "
            f"n_target_dims={_n_target_dims}, n_delays={_n_delays}, "
            f"obs_jac={COMPUTE_OBS_JAC}"
        )

        # Result dict
        _result = {
            "run_id": _run_id,
            "config": {
                "loop_closure_weight": _lc_weight,
                "kl_dyn_weight": _kl_dyn,
                "n_target_dims": _n_target_dims,
            },
            "pred_lyap_per_traj": [],
            "emp_lyap_per_traj": [],
            "spectrum_r2_per_traj": [],
            "spectrum_corr_per_traj": [],
            "obs_jac_lyap_per_traj": [],
            "jac_r2_per_traj": [],
            "timestamp": datetime.now().isoformat(),
        }

        with torch.no_grad():
            # ============================================================
            # 1. Encode all test trajectories at once and slice to dyn dims
            # ============================================================
            print(f"    [encode] encoding {_n_test} test trajectories on {device} ...")
            _t0 = time.perf_counter()
            _traj_t = torch.as_tensor(_test_seq).float().to(device)
            _z_full_all = _lit_model.encode_trajectory(_traj_t)
            _z_dyn_all = z_dyn_slice(_z_full_all, _n_target_dims)  # (n_test, T', D_dyn)
            if device.type == "cuda":
                torch.cuda.synchronize()
            print(
                f"    [encode] z_dyn_all shape={tuple(_z_dyn_all.shape)} "
                f"({time.perf_counter() - _t0:.2f}s)"
            )

            # ============================================================
            # 2. Predicted Lyapunov spectra — chunk-batched on GPU
            # ============================================================
            _n_chunks_pred = (_n_test + _LYAP_CHUNK - 1) // _LYAP_CHUNK
            _pred_le_chunks = []
            _t0 = time.perf_counter()
            for _ci in tqdm(
                range(0, _n_test, _LYAP_CHUNK),
                total=_n_chunks_pred,
                desc="    [pred LE] latent chunks",
                leave=False,
            ):
                _z_chunk = _z_dyn_all[_ci:_ci + _LYAP_CHUNK]
                _jacs_chunk = _lit_model.compute_jacobians(_z_chunk)        # (B,T',D,D) on GPU
                _le_chunk = LitLatentJacobianODE.compute_lyapunov_exponents(
                    _jacs_chunk, _dt_shared
                )                                                            # (B,D) on GPU
                _pred_le_chunks.append(_le_chunk.cpu())
                del _jacs_chunk, _le_chunk
            _pred_le_all = torch.cat(_pred_le_chunks, dim=0)                 # (n_test, D)
            if device.type == "cuda":
                torch.cuda.synchronize()
            print(
                f"    [pred LE] done in {time.perf_counter() - _t0:.2f}s "
                f"(shape={tuple(_pred_le_all.shape)})"
            )

            # ============================================================
            # 3. Empirical Lyapunov spectra — chunk-batched on GPU when possible
            # ============================================================
            # Denormalize once outside of any loop.
            _test_raw_np = np.asarray(_test_seq) * _sigma + _mu              # (n_test, T_full, D_obs)

            # Retain empirical Jacobians on CPU only when needed for obs-jac R^2.
            _retain_emp_jacs = COMPUTE_OBS_JAC and (_n_delays == 1)
            _emp_jacs_cpu_chunks: list[torch.Tensor] = []

            _emp_le_chunks = []
            _is_torch_eq = hasattr(_eq_shared, "model")
            _n_chunks_emp = (_n_test + _LYAP_CHUNK - 1) // _LYAP_CHUNK
            print(
                f"    [emp LE] torch_eq={_is_torch_eq}, retain_jacs={_retain_emp_jacs}"
            )
            _t0 = time.perf_counter()
            for _ci in tqdm(
                range(0, _n_test, _LYAP_CHUNK),
                total=_n_chunks_emp,
                desc="    [emp LE] empirical chunks",
                leave=False,
            ):
                if _is_torch_eq:
                    _traj_chunk = torch.as_tensor(
                        _test_raw_np[_ci:_ci + _LYAP_CHUNK]
                    ).float().to(device)
                    _jacs_true_chunk = _eq_shared.jac(_traj_chunk, t=0)
                    if not torch.is_tensor(_jacs_true_chunk):
                        _jacs_true_chunk = torch.as_tensor(_jacs_true_chunk).float()
                    _jacs_true_chunk = _jacs_true_chunk.to(device).float()
                else:
                    _jacs_list = [
                        np.asarray(_eq_shared.jac(_test_raw_np[_i], t=0))
                        for _i in range(_ci, min(_ci + _LYAP_CHUNK, _n_test))
                    ]
                    _jacs_true_chunk = torch.as_tensor(
                        np.stack(_jacs_list, axis=0)
                    ).float().to(device)

                _le_chunk = LitLatentJacobianODE.compute_lyapunov_exponents(
                    _jacs_true_chunk, _dt_shared
                )
                _emp_le_chunks.append(_le_chunk.cpu())
                if _retain_emp_jacs:
                    _emp_jacs_cpu_chunks.append(_jacs_true_chunk.cpu())
                del _jacs_true_chunk, _le_chunk
            _emp_le_all = torch.cat(_emp_le_chunks, dim=0)                   # (n_test, D)
            _emp_jacs_full_cpu = (
                torch.cat(_emp_jacs_cpu_chunks, dim=0) if _retain_emp_jacs else None
            )
            if device.type == "cuda":
                torch.cuda.synchronize()
            print(
                f"    [emp LE] done in {time.perf_counter() - _t0:.2f}s "
                f"(shape={tuple(_emp_le_all.shape)})"
            )

            # ============================================================
            # 4. Optional: observed-space Jacobian via batched vmap,
            #    then one batched Lyapunov sweep over the stacked result.
            # ============================================================
            _obs_le_all = None
            _obs_success_idx: list[int] = []
            _J_obs_by_idx: dict[int, torch.Tensor] = {}
            if COMPUTE_OBS_JAC:
                print(
                    f"    [obs jac] batched vmap across trajectories "
                    f"(traj_chunk={_OBS_JAC_TRAJ_CHUNK}, enc_chunk={_ENC_JAC_CHUNK}) ..."
                )
                _t0 = time.perf_counter()
                _J_obs_all_chunks: list[torch.Tensor] = []
                _n_traj_chunks = (_n_test + _OBS_JAC_TRAJ_CHUNK - 1) // _OBS_JAC_TRAJ_CHUNK
                _obs_failed_chunk = False
                try:
                    for _ci in tqdm(
                        range(0, _n_test, _OBS_JAC_TRAJ_CHUNK),
                        total=_n_traj_chunks,
                        desc="    [obs jac] traj chunks",
                        leave=False,
                    ):
                        _traj_chunk = _traj_t[_ci:_ci + _OBS_JAC_TRAJ_CHUNK]
                        _J_obs_chunk = compute_observed_space_jacobians_batched(
                            _lit_model,
                            _traj_chunk,
                            _n_target_dims,
                            device,
                            chunk_size=_ENC_JAC_CHUNK,
                        )                                                     # (B_chunk, T', D_obs, D_obs)
                        _J_obs_all_chunks.append(_J_obs_chunk)
                except Exception as _obs_batch_err:
                    _obs_failed_chunk = True
                    print(
                        f"    [obs jac] batched path failed ({_obs_batch_err!r}); "
                        f"falling back to per-trajectory compute"
                    )

                if not _obs_failed_chunk and _J_obs_all_chunks:
                    _J_obs_stack = torch.cat(_J_obs_all_chunks, dim=0)        # (n_test, T', D_obs, D_obs)
                    _obs_success_idx = list(range(_n_test))
                    if _n_delays == 1:
                        for _ti in range(_n_test):
                            _J_obs_by_idx[_ti] = _J_obs_stack[_ti].cpu()
                    if device.type == "cuda":
                        torch.cuda.synchronize()
                    print(
                        f"    [obs jac] batched compute done in {time.perf_counter() - _t0:.2f}s "
                        f"(stacked shape={tuple(_J_obs_stack.shape)})"
                    )
                    _t0 = time.perf_counter()
                    _obs_le_all = LitLatentJacobianODE.compute_lyapunov_exponents(
                        _J_obs_stack, _dt_shared
                    ).cpu()
                    del _J_obs_stack, _J_obs_all_chunks
                    if device.type == "cuda":
                        torch.cuda.synchronize()
                    print(f"    [obs LE] done in {time.perf_counter() - _t0:.2f}s")
                else:
                    # Fallback: per-trajectory compute with the original helper.
                    del _J_obs_all_chunks
                    torch.cuda.empty_cache() if device.type == "cuda" else None
                    _t0 = time.perf_counter()
                    _J_obs_chunks = []
                    for _ti in tqdm(
                        range(_n_test),
                        desc="    [obs jac] trajs (fallback)",
                        leave=False,
                    ):
                        try:
                            _J_obs = compute_observed_space_jacobians(
                                _lit_model,
                                _traj_t[_ti],
                                _n_target_dims,
                                device,
                            )                                                 # (T', D_obs, D_obs)
                            _J_obs_chunks.append(_J_obs)
                            _obs_success_idx.append(_ti)
                            if _n_delays == 1:
                                _J_obs_by_idx[_ti] = _J_obs.cpu()
                        except Exception as _obs_err:
                            print(f"      traj {_ti} failed: {_obs_err}")
                    if device.type == "cuda":
                        torch.cuda.synchronize()
                    print(
                        f"    [obs jac] per-traj compute done in {time.perf_counter() - _t0:.2f}s "
                        f"({len(_obs_success_idx)}/{_n_test} ok)"
                    )
                    if _J_obs_chunks:
                        _t0 = time.perf_counter()
                        _J_obs_stack = torch.stack(_J_obs_chunks, dim=0)
                        _obs_le_all = LitLatentJacobianODE.compute_lyapunov_exponents(
                            _J_obs_stack, _dt_shared
                        ).cpu()
                        del _J_obs_stack, _J_obs_chunks
                        if device.type == "cuda":
                            torch.cuda.synchronize()
                        print(f"    [obs LE] done in {time.perf_counter() - _t0:.2f}s")

            # ============================================================
            # 5. Per-trajectory result population (no GPU kernels here)
            # ============================================================
            _pred_np_full = _pred_le_all.numpy()
            _emp_np_full = _emp_le_all.numpy()
            for _ti in range(_n_test):
                _pred_le_np = _pred_np_full[_ti]
                _emp_le_np = _emp_np_full[_ti]
                _corr, _r2 = spectrum_comparison(_pred_le_np, _emp_le_np)

                _result["pred_lyap_per_traj"].append(_pred_le_np.tolist())
                _result["emp_lyap_per_traj"].append(_emp_le_np.tolist())
                _result["spectrum_r2_per_traj"].append(_r2)
                _result["spectrum_corr_per_traj"].append(_corr)

                if _ti < 3 or _ti == _n_test - 1:
                    print(
                        f"    Traj {_ti}: pred_LE={np.array(_pred_le_np)[:3]}, "
                        f"r2={_r2:.4f}"
                    )

            if _obs_le_all is not None:
                _obs_np_full = _obs_le_all.numpy()
                # _obs_le_all rows are in the same order as _obs_success_idx.
                for _ok, _ti in enumerate(_obs_success_idx):
                    _result["obs_jac_lyap_per_traj"].append(
                        _obs_np_full[_ok].tolist()
                    )
                    if _n_delays == 1 and _emp_jacs_full_cpu is not None:
                        _J_obs_ti = _J_obs_by_idx[_ti]
                        _jacs_true_ti = _emp_jacs_full_cpu[_ti]
                        _T_cmp = min(_J_obs_ti.shape[0], _jacs_true_ti.shape[0])
                        _jac_r2 = float(
                            r2_score(
                                _jacs_true_ti[:_T_cmp].reshape(-1),
                                _J_obs_ti[:_T_cmp].reshape(-1),
                            )
                        )
                        _result["jac_r2_per_traj"].append(_jac_r2)

        # -- Aggregate stats --
        _pred_arr = np.array(_result["pred_lyap_per_traj"])
        _emp_arr = np.array(_result["emp_lyap_per_traj"])
        _result["pred_lyap_mean"] = _pred_arr.mean(axis=0).tolist()
        _result["pred_lyap_std"] = _pred_arr.std(axis=0).tolist()
        _result["emp_lyap_mean"] = _emp_arr.mean(axis=0).tolist()
        _result["emp_lyap_std"] = _emp_arr.std(axis=0).tolist()
        _result["mean_spectrum_r2"] = float(np.nanmean(_result["spectrum_r2_per_traj"]))
        _result["mean_spectrum_corr"] = float(np.nanmean(_result["spectrum_corr_per_traj"]))

        if _result["obs_jac_lyap_per_traj"]:
            _obs_arr = np.array(_result["obs_jac_lyap_per_traj"])
            _result["obs_jac_lyap_mean"] = _obs_arr.mean(axis=0).tolist()
            _result["obs_jac_lyap_std"] = _obs_arr.std(axis=0).tolist()
        if _result["jac_r2_per_traj"]:
            _result["mean_jac_r2"] = float(np.nanmean(_result["jac_r2_per_traj"]))

        # Save to cache
        save_results(_cache_path, _result)
        all_results[_run_id] = _result

        print(
            f"  [{_ri+1}/{len(discovered.run_ids)}] {_run_id}: "
            f"mean_r2={_result['mean_spectrum_r2']:.4f}, "
            f"mean_corr={_result['mean_spectrum_corr']:.4f}"
        )

        # Cleanup
        _lit_model.cpu()
        del _lit_model
        torch.cuda.empty_cache()

    except Exception as _err:
        all_results[_run_id] = {
            "run_id": _run_id,
            "error": str(_err),
            "traceback": traceback.format_exc(),
        }
        print(
            f"  [{_ri+1}/{len(discovered.run_ids)}] {_run_id}: ERROR - {_err}"
        )

print(f"Done. Processed {len(all_results)} runs. Results cached in `{results_dir}`.")

Processing runs:   0%|          | 0/79 [00:00<?, ?it/s]

  [1/79] lhlqgosq: loaded from cache
  [2/79] 4gi2m7hl: loaded from cache
  [3/79] gw47jqdi: loaded from cache
  [4/79] 1hl94h34: loaded from cache
  [5/79] 3faysy6m: loaded from cache
  [6/79] mfxypvv4: loaded from cache
  [7/79] dvirdh5p: loaded from cache
  [8/79] sxj3tx2n: loaded from cache
  [9/79] vuh0ejh1: loaded from cache
  [10/79] uaqrbwbe: loaded from cache
  [11/79] eyv5emvs: loaded from cache
  [12/79] ij7nctt5: loaded from cache
  [13/79] o61uqaqh: loaded from cache
  [14/79] v2y903yb: recomputing ...
    [load] loading run + model ...
loaded wmtask RNN model checkpoint 41
    [config] n_test=409, T=49, D_obs=128, n_target_dims=128, n_delays=1, obs_jac=True
    [encode] encoding 409 test trajectories on cuda ...
    [encode] z_dyn_all shape=(409, 49, 128) (0.23s)


    [pred LE] latent chunks:   0%|          | 0/7 [00:00<?, ?it/s]

    [pred LE] done in 5.97s (shape=(409, 128))
    [emp LE] torch_eq=True, retain_jacs=True


    [emp LE] empirical chunks:   0%|          | 0/7 [00:00<?, ?it/s]

...

    [emp LE] empirical chunks:   0%|          | 0/7 [00:00<?, ?it/s]

    [emp LE] done in 6.08s (shape=(409, 128))
    [obs jac] batched vmap across trajectories (traj_chunk=32, enc_chunk=256) ...


    [obs jac] traj chunks:   0%|          | 0/13 [00:00<?, ?it/s]

    [obs jac] batched compute done in 120.24s (stacked shape=(409, 49, 128, 128))
    [obs LE] done in 2.14s
    Traj 0: pred_LE=[-0.4444433  -0.46428475 -0.523576  ], r2=-2.2153
    Traj 1: pred_LE=[-0.5647871 -0.7387672 -0.9117865], r2=-1.7360
    Traj 2: pred_LE=[-0.5040504  -0.7191436  -0.90985984], r2=-2.2433
    Traj 408: pred_LE=[-0.07331374 -0.3328957  -0.699914  ], r2=-2.2507
  [29/79] 16j5ith4: mean_r2=-2.1219, mean_corr=0.9274
  [30/79] qrl0xy0b: recomputing ...
    [load] loading run + model ...
    [config] n_test=409, T=49, D_obs=128, n_target_dims=128, n_delays=1, obs_jac=True
    [encode] encoding 409 test trajectories on cuda ...
    [encode] z_dyn_all shape=(409, 49, 128) (0.24s)


    [pred LE] latent chunks:   0%|          | 0/7 [00:00<?, ?it/s]

## Summary Table

In [ ]:
# Summary table
# Per-trajectory mean R²/MSE and R²/MSE of mean spectra, at full / top-10 / top-5
# In both exponent space and timescale space (1/|lambda|).

def partial_metric_per_traj(pred_per_traj, emp_per_traj, k, metric_fn):
    """Mean of per-trajectory metric using only first k exponents."""
    vals = []
    for p, e in zip(pred_per_traj, emp_per_traj):
        pa, ea = np.array(p), np.array(e)
        n = min(len(pa), len(ea), k)
        if n == 0:
            continue
        vals.append(metric_fn(pa[:n], ea[:n]))
    return float(np.nanmean(vals)) if vals else None

def r2_fn(p, e):
    return spectrum_comparison(p, e)[1]

def ts_r2_fn(p, e):
    return spectrum_comparison(lyap_to_timescale(p), lyap_to_timescale(e))[1]

def mse_fn(p, e):
    return spectrum_mse(p, e)

def ts_mse_fn(p, e):
    return spectrum_mse(lyap_to_timescale(p), lyap_to_timescale(e))

_rows = []
for _run_id in discovered.run_ids:
    _r = all_results.get(_run_id, {})
    _wm = wandb_val_metrics.get(_run_id, {})

    if "error" in _r:
        _rows.append({"run_id": _run_id, "error": _r.get("error", "unknown")})
        continue

    _cfg = _r.get("config", {})
    _pred_mean = np.array(_r.get("pred_lyap_mean", []))
    _emp_mean = np.array(_r.get("emp_lyap_mean", []))
    _pred_per = _r.get("pred_lyap_per_traj", [])
    _emp_per = _r.get("emp_lyap_per_traj", [])
    _n_le = min(len(_pred_mean), len(_emp_mean))

    # Timescale versions of mean spectra
    _pred_ts_mean = lyap_to_timescale(_pred_mean)
    _emp_ts_mean = lyap_to_timescale(_emp_mean)

    _row = {
        "run_id": _run_id,
        "lambda_lc": _cfg.get("loop_closure_weight"),
        "kl_dyn": _cfg.get("kl_dyn_weight"),
        "traj_val_loss": _wm.get("trajectory_val_loss"),
        "val_lc_loss": _wm.get("val_loop_closure_loss"),
    }

    # --- Exponent-space metrics ---
    _, _row["ms_r2_full"] = spectrum_comparison(_pred_mean, _emp_mean)
    _row["ms_mse_full"] = spectrum_mse(_pred_mean, _emp_mean)
    _row["pt_r2_full"] = _r.get("mean_spectrum_r2")
    _row["pt_mse_full"] = partial_metric_per_traj(_pred_per, _emp_per, _n_le, mse_fn)

    if _n_le >= 5:
        _row["ms_r2_5"] = spectrum_comparison(_pred_mean[:5], _emp_mean[:5])[1]
        _row["ms_mse_5"] = spectrum_mse(_pred_mean[:5], _emp_mean[:5])
        _row["pt_r2_5"] = partial_metric_per_traj(_pred_per, _emp_per, 5, r2_fn)
        _row["pt_mse_5"] = partial_metric_per_traj(_pred_per, _emp_per, 5, mse_fn)
    if _n_le >= 10:
        _row["ms_r2_10"] = spectrum_comparison(_pred_mean[:10], _emp_mean[:10])[1]
        _row["ms_mse_10"] = spectrum_mse(_pred_mean[:10], _emp_mean[:10])
        _row["pt_r2_10"] = partial_metric_per_traj(_pred_per, _emp_per, 10, r2_fn)
        _row["pt_mse_10"] = partial_metric_per_traj(_pred_per, _emp_per, 10, mse_fn)

    # --- Timescale-space metrics ---
    _, _row["ts_ms_r2_full"] = spectrum_comparison(_pred_ts_mean, _emp_ts_mean)
    _row["ts_ms_mse_full"] = spectrum_mse(_pred_ts_mean, _emp_ts_mean)
    _row["ts_pt_r2_full"] = partial_metric_per_traj(_pred_per, _emp_per, _n_le, ts_r2_fn)
    _row["ts_pt_mse_full"] = partial_metric_per_traj(_pred_per, _emp_per, _n_le, ts_mse_fn)

    if _n_le >= 5:
        _row["ts_ms_r2_5"] = spectrum_comparison(_pred_ts_mean[:5], _emp_ts_mean[:5])[1]
        _row["ts_ms_mse_5"] = spectrum_mse(_pred_ts_mean[:5], _emp_ts_mean[:5])
        _row["ts_pt_r2_5"] = partial_metric_per_traj(_pred_per, _emp_per, 5, ts_r2_fn)
        _row["ts_pt_mse_5"] = partial_metric_per_traj(_pred_per, _emp_per, 5, ts_mse_fn)
    if _n_le >= 10:
        _row["ts_ms_r2_10"] = spectrum_comparison(_pred_ts_mean[:10], _emp_ts_mean[:10])[1]
        _row["ts_ms_mse_10"] = spectrum_mse(_pred_ts_mean[:10], _emp_ts_mean[:10])
        _row["ts_pt_r2_10"] = partial_metric_per_traj(_pred_per, _emp_per, 10, ts_r2_fn)
        _row["ts_pt_mse_10"] = partial_metric_per_traj(_pred_per, _emp_per, 10, ts_mse_fn)

    # --- Jacobian metrics ---
    _row["mean_jac_r2"] = _r.get("mean_jac_r2")
    _row["mean_jac_mse"] = None  # would need recomputation; not cached

    _row["pred_LE_1"] = float(_pred_mean[0]) if len(_pred_mean) > 0 else None
    _row["emp_LE_1"] = float(_emp_mean[0]) if len(_emp_mean) > 0 else None
    _row["error"] = None
    _rows.append(_row)

summary_df = pd.DataFrame(_rows)
summary_df

,run_id,lambda_lc,kl_dyn,traj_val_loss,val_lc_loss,ms_r2_full,ms_mse_full,pt_r2_full,pt_mse_full,ms_r2_5,...,ts_pt_mse_5,ts_ms_r2_10,ts_ms_mse_10,ts_pt_r2_10,ts_pt_mse_10,mean_jac_r2,mean_jac_mse,pred_LE_1,emp_LE_1,error
0,lhlqgosq,0.0,0.000000,0.005197,9.793925,-1.174939,201.277370,-1.197113,202.709085,0.813124,...,44.243420,0.866928,0.018556,-43.837299,22.123613,0.386350,None,-0.926270,-0.679087,None
1,4gi2m7hl,0.0,0.000001,0.005024,9.885106,-1.059841,190.625753,-1.082539,192.116125,0.868578,...,65.793515,0.866504,0.018615,-197.613120,32.898106,0.396948,None,-0.937303,-0.679087,None
2,gw47jqdi,0.0,0.000010,0.00505,9.394964,-1.058741,190.523873,-1.080155,192.014645,0.904027,...,78.538098,0.902590,0.013583,-69.731739,39.270323,0.396760,None,-0.887534,-0.679087,None
3,1hl94h34,0.0,0.000100,0.005074,12.603946,-1.616236,242.116657,-1.630402,243.123560,0.924500,...,1517.104412,0.853807,0.020385,-4421.376391,758.553018,0.357744,None,-0.943811,-0.679087,None
4,3faysy6m,0.0,0.001000,0.004784,17.070967,-1.046307,189.373256,-1.058697,190.271565,-0.179640,...,38.976291,0.322556,0.094462,0.296175,19.490041,0.445126,None,-1.679714,-0.679087,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,3otjq9ns,NaN,NaN,0.02077,0.000001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,None
75,h6i12ssy,NaN,NaN,0.02133,0.000002,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,None
76,hr0tnaqh,NaN,NaN,0.022124,0.000004,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,None
77,dcl3nyez,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,None


## Per-Run Lyapunov Spectrum Plots

In [ ]:
# Per-run Lyapunov spectrum plots
lyap_figs = {}
for _run_id in discovered.run_ids:
    _r = all_results.get(_run_id, {})
    if "error" in _r or "pred_lyap_mean" not in _r:
        continue

    _pred_mean = np.array(_r["pred_lyap_mean"])
    _pred_std = np.array(_r["pred_lyap_std"])
    _emp_mean = np.array(_r["emp_lyap_mean"])
    _emp_std = np.array(_r["emp_lyap_std"])
    _lc = _r.get("config", {}).get("loop_closure_weight")

    _result = plot_lyapunov_spectrum(
        _pred_mean,
        _pred_std,
        _emp_mean,
        _emp_std,
        true_lyapunov=TRUE_LYAPUNOV,
        loop_closure_weight=_lc,
    )
    if isinstance(_result, list):
        _fig = _result[0]
    else:
        _fig = _result
    _fig.suptitle(f"Run {_run_id} (lc={_lc})", fontsize=10, y=1.02)
    lyap_figs[_run_id] = _fig

print(f"{len(lyap_figs)} Lyapunov spectrum plots generated." if lyap_figs else "No successful runs to plot.")

/orcd/home/002/eisenaj/code/JacobianODE/JacobianODE/jacobians/run_analytics.py:497: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig_all, ax_all = plt.subplots(figsize=(14, 5))


75 Lyapunov spectrum plots generated.


## Cross-Run Comparison Plots

In [ ]:
# Cross-run comparison plots
_lcs = []
_r2s = []
_pred_le1s = []
_emp_le1s = []
_labels = []

for _run_id in discovered.run_ids:
    _r = all_results.get(_run_id, {})
    if "error" in _r or "mean_spectrum_r2" not in _r:
        continue
    _cfg = _r.get("config", {})
    _lcs.append(_cfg.get("loop_closure_weight", 0))
    _r2s.append(_r["mean_spectrum_r2"])
    _pred = _r.get("pred_lyap_mean", [])
    _emp = _r.get("emp_lyap_mean", [])
    _pred_le1s.append(_pred[0] if _pred else float("nan"))
    _emp_le1s.append(_emp[0] if _emp else float("nan"))
    _labels.append(_run_id[:8])

if _lcs:
    cross_fig, (_ax1, _ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Largest LE comparison
    _ax1.scatter(_emp_le1s, _pred_le1s, c=np.log10(np.array(_lcs) + 1e-10), cmap="viridis", s=60)
    _lim = [
        min(min(_emp_le1s), min(_pred_le1s)) - 0.1,
        max(max(_emp_le1s), max(_pred_le1s)) + 0.1,
    ]
    _ax1.plot(_lim, _lim, "k--", alpha=0.3, label="y=x")
    _ax1.set_xlabel("Empirical LE_1")
    _ax1.set_ylabel("Predicted LE_1")
    _ax1.set_title("Largest Lyapunov Exponent")
    _ax1.legend()

    # Right: R^2 vs loop_closure_weight
    _ax2.scatter(_lcs, _r2s, s=60, c="steelblue")
    _ax2.set_xscale("symlog", linthresh=1e-7)
    _ax2.set_xlabel("loop_closure_weight")
    _ax2.set_ylabel("Mean Spectrum R^2")
    _ax2.set_title("Spectrum R^2 vs Loop Closure Weight")
    _ax2.axhline(y=1.0, color="k", linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    cross_fig = None
    print("No runs to compare.")

## Scatter Plots: R²/MSE Metrics vs Validation Losses

In [ ]:
# Scatter plots: R²/MSE metrics vs val losses
_df = summary_df.dropna(subset=["lambda_lc"]).copy()

# Add n_target_dims from cached results config
_df["n_target_dims"] = _df["run_id"].map(
    lambda rid: all_results.get(rid, {}).get("config", {}).get("n_target_dims")
)

def collect_available(col_label_pairs, df):
    """Filter to columns that exist and have data."""
    return [(c, l) for c, l in col_label_pairs if c in df.columns and df[c].notna().any()]

# -- Exponent-space R² columns --
_exp_r2_cols = collect_available([
    ("pt_r2_5", "Per-Traj R² (top 5)"),
    ("pt_r2_10", "Per-Traj R² (top 10)"),
    ("pt_r2_full", "Per-Traj R² (full)"),
    ("ms_r2_5", "Mean-Spec R² (top 5)"),
    ("ms_r2_10", "Mean-Spec R² (top 10)"),
    ("ms_r2_full", "Mean-Spec R² (full)"),
    ("mean_jac_r2", "Jacobian R²"),
], _df)

# -- Exponent-space MSE columns --
_exp_mse_cols = collect_available([
    ("pt_mse_5", "Per-Traj MSE (top 5)"),
    ("pt_mse_10", "Per-Traj MSE (top 10)"),
    ("pt_mse_full", "Per-Traj MSE (full)"),
    ("ms_mse_5", "Mean-Spec MSE (top 5)"),
    ("ms_mse_10", "Mean-Spec MSE (top 10)"),
    ("ms_mse_full", "Mean-Spec MSE (full)"),
], _df)

# -- Timescale-space R² columns --
_ts_r2_cols = collect_available([
    ("ts_pt_r2_5", "TS Per-Traj R² (top 5)"),
    ("ts_pt_r2_10", "TS Per-Traj R² (top 10)"),
    ("ts_pt_r2_full", "TS Per-Traj R² (full)"),
    ("ts_ms_r2_5", "TS Mean-Spec R² (top 5)"),
    ("ts_ms_r2_10", "TS Mean-Spec R² (top 10)"),
    ("ts_ms_r2_full", "TS Mean-Spec R² (full)"),
], _df)

# -- Timescale-space MSE columns --
_ts_mse_cols = collect_available([
    ("ts_pt_mse_5", "TS Per-Traj MSE (top 5)"),
    ("ts_pt_mse_10", "TS Per-Traj MSE (top 10)"),
    ("ts_pt_mse_full", "TS Per-Traj MSE (full)"),
    ("ts_ms_mse_5", "TS Mean-Spec MSE (top 5)"),
    ("ts_ms_mse_10", "TS Mean-Spec MSE (top 10)"),
    ("ts_ms_mse_full", "TS Mean-Spec MSE (full)"),
], _df)

def make_scatter_grid(df, x_col, x_label, title_prefix, metric_cols,
                      y_label="R²", y_clip=(-15, 1.1), log_x=False, log_y=False):
    """Scatter subplots: each metric vs x_col."""
    _n = len(metric_cols)
    if _n == 0:
        return None
    _ncols = min(4, _n)
    _nrows = (_n + _ncols - 1) // _ncols
    fig, axes = plt.subplots(_nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), squeeze=False)

    for _idx, (_col, _label) in enumerate(metric_cols):
        _ax = axes[_idx // _ncols][_idx % _ncols]
        _mask = df[_col].notna() & df[x_col].notna()
        _x = df.loc[_mask, x_col].values
        _y = df.loc[_mask, _col].values
        if not log_y:
            _y = np.clip(_y, y_clip[0], y_clip[1])

        _ax.scatter(_x, _y, s=25, alpha=0.7, c="steelblue", edgecolors="k", linewidths=0.3)
        if not log_y:
            _ax.axhline(y=1.0, color="k", linestyle="--", alpha=0.3, linewidth=0.8)
            _ax.axhline(y=0.0, color="gray", linestyle=":", alpha=0.3, linewidth=0.8)
            _ax.set_ylim(y_clip)
        if log_x:
            _ax.set_xscale("log")
        if log_y:
            _ax.set_yscale("log")
        _ax.set_xlabel(x_label, fontsize=9)
        _ax.set_ylabel(y_label, fontsize=9)
        _ax.set_title(_label, fontsize=10)

    for _idx2 in range(_n, _nrows * _ncols):
        axes[_idx2 // _ncols][_idx2 % _ncols].set_visible(False)

    fig.suptitle(title_prefix, fontsize=13, y=1.02)
    plt.tight_layout()
    return fig

scatter_figs = {}

if _df.empty:
    print("No data for scatter plots.")
else:
    _metric_groups = [
        ("exp_r2", _exp_r2_cols, "Exponent R²", "R²", (-15, 1.1), False),
        ("exp_mse", _exp_mse_cols, "Exponent MSE", "MSE", None, True),
        ("ts_r2", _ts_r2_cols, "Timescale R²", "R²", (-15, 1.1), False),
        ("ts_mse", _ts_mse_cols, "Timescale MSE", "MSE", None, True),
    ]

    _x_groups = []

    # Group 1: all runs vs traj val loss
    _df1 = _df.dropna(subset=["traj_val_loss"])
    if not _df1.empty:
        _x_groups.append(("traj_vloss", _df1, "traj_val_loss", "trajectory val loss", "all runs"))

    # Group 2: all runs vs LC val loss
    _df2 = _df.dropna(subset=["val_lc_loss"])
    if not _df2.empty:
        _x_groups.append(("lc_vloss", _df2, "val_lc_loss", "loop closure val loss", "all runs"))

    # Group 3: feasible runs vs traj val loss
    _df3 = _df.dropna(subset=["traj_val_loss", "val_lc_loss", "n_target_dims"]).copy()
    if not _df3.empty:
        _df3 = _df3[_df3["val_lc_loss"] <= np.sqrt(_df3["n_target_dims"])]
        if not _df3.empty:
            _x_groups.append(("traj_vloss_feasible", _df3, "traj_val_loss", "trajectory val loss",
                              f"feasible (n={len(_df3)})"))

    for _xkey, _xdf, _xcol, _xlabel, _xsuffix in _x_groups:
        for _mkey, _mcols, _mtitle, _ylabel, _yclip, _logy in _metric_groups:
            if not _mcols:
                continue
            _fig = make_scatter_grid(
                _xdf, _xcol, _xlabel,
                f"{_mtitle} vs {_xlabel} ({_xsuffix})",
                _mcols, y_label=_ylabel, y_clip=_yclip, log_x=True, log_y=_logy,
            )
            if _fig is not None:
                scatter_figs[f"{_mkey}_vs_{_xkey}"] = _fig

    print(f"Generated {len(scatter_figs)} scatter plot grids.")
    plt.show()

## HTML Report Generation

In [ ]:
# HTML report generation

_CSS = """
body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
       max-width: 1200px; margin: 0 auto; padding: 24px 32px; background: #fff; color: #222; }
h1   { font-size: 1.6em; margin-bottom: 0.15em; }
h2   { font-size: 1.15em; color: #444; border-bottom: 2px solid #e0e0e0;
        padding-bottom: 4px; margin-top: 2.2em; }
pre  { background: #f6f8fa; border: 1px solid #e1e4e8; border-radius: 6px;
        padding: 12px 16px; font-size: 12.5px; line-height: 1.6;
        overflow-x: auto; white-space: pre-wrap; word-break: break-word; }
.figure { margin: 18px 0; text-align: center; }
img  { max-width: 100%; border: 1px solid #e0e0e0; border-radius: 4px; }
.subtitle { color: #666; font-size: 0.9em; margin-bottom: 2em; }
table { border-collapse: collapse; margin: 12px 0; }
th, td { border: 1px solid #e0e0e0; padding: 6px 12px; text-align: left; font-size: 13px; }
th { background: #f6f8fa; }
"""

def make_html_report(title, subtitle, sections):
    """Build a self-contained HTML report.

    sections: list of (heading, content_html) tuples
    """
    body_parts = []
    for heading, content in sections:
        body_parts.append(f"<h2>{heading}</h2>\n{content}")
    body = "\n".join(body_parts)
    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{title}</title>
  <style>{_CSS}</style>
</head>
<body>
<h1>{title}</h1>
<p class="subtitle">{subtitle}</p>
{body}
</body>
</html>"""

def fig_html(fig_or_b64):
    if isinstance(fig_or_b64, str):
        b64 = fig_or_b64
    else:
        b64 = fig_to_base64(fig_or_b64)
    return f'<div class="figure"><img src="data:image/png;base64,{b64}"/></div>'

def table_html(df):
    return df.to_html(index=False, na_rep="--", float_format=lambda x: f"{x:.4f}")

_repo_root = Path.cwd().parent if Path.cwd().name == "_notebook" else Path.cwd()

# --- Per-run HTML reports ---
_per_run_dir = results_dir
for _run_id in discovered.run_ids:
    _r = all_results.get(_run_id, {})
    if "error" in _r:
        continue
    _cfg = _r.get("config", {})
    _sections = []

    # Config section
    _config_lines = "\n".join(f"{k}: {v}" for k, v in _cfg.items())
    _sections.append(("Configuration", f"<pre>{_config_lines}</pre>"))

    # Lyapunov spectrum
    _lyap_lines = []
    _pred_mean = _r.get("pred_lyap_mean", [])
    _pred_std = _r.get("pred_lyap_std", [])
    _emp_mean = _r.get("emp_lyap_mean", [])
    _emp_std = _r.get("emp_lyap_std", [])
    _lyap_lines.append("Predicted (mean +/- std):")
    for _i, (_le, _sd) in enumerate(zip(_pred_mean, _pred_std)):
        _lyap_lines.append(f"  LE_{_i+1} = {_le:+.4f} +/- {_sd:.4f}")
    _lyap_lines.append("Empirical (mean +/- std):")
    for _i, (_le, _sd) in enumerate(zip(_emp_mean, _emp_std)):
        _lyap_lines.append(f"  LE_{_i+1} = {_le:+.4f} +/- {_sd:.4f}")
    _lyap_lines.append(f"Mean R^2:   {_r.get('mean_spectrum_r2', float('nan')):.4f}")
    _lyap_lines.append(f"Mean corr:  {_r.get('mean_spectrum_corr', float('nan')):.4f}")
    _content = "<pre>" + "\n".join(_lyap_lines) + "</pre>"
    if _run_id in lyap_figs:
        _content += fig_html(fig_to_base64(lyap_figs[_run_id]))
    _sections.append(("Lyapunov Spectrum", _content))

    # Obs-space Jacobian (if computed)
    if _r.get("obs_jac_lyap_mean"):
        _obs_lines = ["Observed-space Jacobian Lyapunov (mean +/- std):"]
        for _i, (_le, _sd) in enumerate(
            zip(_r["obs_jac_lyap_mean"], _r.get("obs_jac_lyap_std", []))
        ):
            _obs_lines.append(f"  LE_{_i+1} = {_le:+.4f} +/- {_sd:.4f}")
        if "mean_jac_r2" in _r:
            _obs_lines.append(f"Mean Jacobian R^2: {_r['mean_jac_r2']:.4f}")
        _sections.append(
            ("Observed-Space Jacobian", "<pre>" + "\n".join(_obs_lines) + "</pre>")
        )

    _html = make_html_report(
        f"Lyapunov Report - {_run_id}",
        f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
        f"Project: {PROJECT} | Group: {GROUP or '(none)'}",
        _sections,
    )
    (_per_run_dir / f"{_run_id}.html").write_text(_html, encoding="utf-8")

# --- Aggregate HTML report ---
_report_dir = _repo_root / "_marimo" / "reports"
_report_dir.mkdir(parents=True, exist_ok=True)
_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
_agg_path = _report_dir / f"{_timestamp} - Lyapunov_{PROJECT}_{GROUP or 'all'}.html"

_agg_sections = []

# Summary table (sorted by full mean-spectrum R² descending)
_sorted_df = summary_df.sort_values("ms_r2_full", ascending=False, na_position="last")
_agg_sections.append(("Summary", table_html(_sorted_df)))

# Cross-run plot
if cross_fig is not None:
    _agg_sections.append(("Cross-Run Comparison", fig_html(fig_to_base64(cross_fig))))

# Scatter plots (all generated grids, in sorted key order)
for _key in sorted(scatter_figs.keys()):
    _fig = scatter_figs[_key]
    if _fig is not None:
        _agg_sections.append((_key.replace("_", " ").title(), fig_html(fig_to_base64(_fig))))

# Per-run spectra
for _run_id in discovered.run_ids:
    if _run_id in lyap_figs:
        _r = all_results.get(_run_id, {})
        _r2 = _r.get("mean_spectrum_r2", float("nan"))
        _agg_sections.append(
            (
                f"Run {_run_id} (R^2={_r2:.4f})",
                fig_html(fig_to_base64(lyap_figs[_run_id])),
            )
        )

_agg_html = make_html_report(
    f"Lyapunov Spectrum Analysis - {PROJECT}",
    f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
    f"Entity: {ENTITY} | Project: {PROJECT} | Group: {GROUP or '(none)'} | "
    f"Runs: {len(discovered.run_ids)}",
    _agg_sections,
)
_agg_path.write_text(_agg_html, encoding="utf-8")

print(
    f"Reports saved.\n"
    f"  Per-run HTML: {_per_run_dir}/\n"
    f"  Aggregate: {_agg_path}"
)

## Exploratory Analysis

In [ ]:
summary_df.columns

In [ ]:
summary_df.loc[summary_df['traj_val_loss'].argsort()]

In [ ]:
valid_runs = summary_df['val_lc_loss'] <= np.sqrt(0.5)
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

thresh = 3
# ms_r2_5
valid_runs_ax0 = valid_runs & ((1 - summary_df['ms_r2_5']) < thresh)
axs[0][0].scatter(summary_df['traj_val_loss'][valid_runs_ax0], 1 - summary_df['ms_r2_5'][valid_runs_ax0])
min_traj_val_idx_0 = summary_df['traj_val_loss'][valid_runs_ax0].idxmin()
axs[0][0].scatter(
    summary_df.loc[min_traj_val_idx_0, 'traj_val_loss'], 
    1 - summary_df.loc[min_traj_val_idx_0, 'ms_r2_5'],
    color='magenta', s=100, label='Min Traj Val Loss', edgecolor='k', zorder=10, alpha=0.7
)
min_nmse_idx_0 = (1 - summary_df['ms_r2_5'][valid_runs_ax0]).idxmin()
axs[0][0].scatter(
    summary_df.loc[min_nmse_idx_0, 'traj_val_loss'], 
    1 - summary_df.loc[min_nmse_idx_0, 'ms_r2_5'],
    color='orange', s=100, label='Min NMSE', edgecolor='k', zorder=10, alpha=0.7
)
axs[0][0].set_xscale('log')
axs[0][0].set_yscale('log')
axs[0][0].set_xlabel('Trajectory Validation Loss')
axs[0][0].set_ylabel('Mean Spectrum\nNormalized MSE\n(first 5 exponents)')
axs[0][0].legend()

# ms_r2_full
valid_runs_ax1 = valid_runs & ((1 - summary_df['ms_r2_full']) < thresh)
axs[0][1].scatter(summary_df['traj_val_loss'][valid_runs_ax1], 1 - summary_df['ms_r2_full'][valid_runs_ax1])
min_traj_val_idx_1 = summary_df['traj_val_loss'][valid_runs_ax1].idxmin()
axs[0][1].scatter(
    summary_df.loc[min_traj_val_idx_1, 'traj_val_loss'], 
    1 - summary_df.loc[min_traj_val_idx_1, 'ms_r2_full'], 
    color='magenta', s=100, label='Min Traj Val Loss', edgecolor='k', zorder=10, alpha=0.7
)
min_nmse_idx_1 = (1 - summary_df['ms_r2_full'][valid_runs_ax1]).idxmin()
axs[0][1].scatter(
    summary_df.loc[min_nmse_idx_1, 'traj_val_loss'], 
    1 - summary_df.loc[min_nmse_idx_1, 'ms_r2_full'],
    color='orange', s=100, label='Min NMSE', edgecolor='k', zorder=10, alpha=0.7
)
axs[0][1].set_xscale('log')
axs[0][1].set_yscale('log')
axs[0][1].set_xlabel('Trajectory Validation Loss')
axs[0][1].set_ylabel('Mean Spectrum\nNormalized MSE\n(full spectrum)')
axs[0][1].legend()

# ts_ms_r2_5
valid_runs_ax2 = valid_runs & ((1 - summary_df['ts_ms_r2_5']) < thresh)
axs[1][0].scatter(summary_df['traj_val_loss'][valid_runs_ax2], 1 - summary_df['ts_ms_r2_5'][valid_runs_ax2])
min_traj_val_idx_2 = summary_df['traj_val_loss'][valid_runs_ax2].idxmin()
axs[1][0].scatter(
    summary_df.loc[min_traj_val_idx_2, 'traj_val_loss'], 
    1 - summary_df.loc[min_traj_val_idx_2, 'ts_ms_r2_5'], 
    color='magenta', s=100, label='Min Traj Val Loss', edgecolor='k', zorder=10, alpha=0.7
)
min_nmse_idx_2 = (1 - summary_df['ts_ms_r2_5'][valid_runs_ax2]).idxmin()
axs[1][0].scatter(
    summary_df.loc[min_nmse_idx_2, 'traj_val_loss'], 
    1 - summary_df.loc[min_nmse_idx_2, 'ts_ms_r2_5'],
    color='orange', s=100, label='Min NMSE', edgecolor='k', zorder=10, alpha=0.7
)
axs[1][0].set_xscale('log')
axs[1][0].set_yscale('log')
axs[1][0].set_xlabel('Trajectory Validation Loss')
axs[1][0].set_ylabel('Time Series Mean Spectrum\nNormalized MSE\n(first 5 exponents)')
axs[1][0].legend()

# ts_ms_r2_full
valid_runs_ax3 = valid_runs & ((1 - summary_df['ts_ms_r2_full']) < thresh)
axs[1][1].scatter(summary_df['traj_val_loss'][valid_runs_ax3], 1 - summary_df['ts_ms_r2_full'][valid_runs_ax3])
min_traj_val_idx_3 = summary_df['traj_val_loss'][valid_runs_ax3].idxmin()
axs[1][1].scatter(
    summary_df.loc[min_traj_val_idx_3, 'traj_val_loss'], 
    1 - summary_df.loc[min_traj_val_idx_3, 'ts_ms_r2_full'], 
    color='magenta', s=100, label='Min Traj Val Loss', edgecolor='k', zorder=10, alpha=0.7
)
min_nmse_idx_3 = (1 - summary_df['ts_ms_r2_full'][valid_runs_ax3]).idxmin()
axs[1][1].scatter(
    summary_df.loc[min_nmse_idx_3, 'traj_val_loss'], 
    1 - summary_df.loc[min_nmse_idx_3, 'ts_ms_r2_full'],
    color='orange', s=100, label='Min NMSE', edgecolor='k', zorder=10, alpha=0.7
)
axs[1][1].set_xscale('log')
axs[1][1].set_yscale('log')
axs[1][1].set_xlabel('Trajectory Validation Loss')
axs[1][1].set_ylabel('Time Series Mean Spectrum\nNormalized MSE\n(full spectrum)')
axs[1][1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cross-metric NMSE scatter (log-log)
valid_runs_nmse_cross = summary_df["val_lc_loss"] <= np.sqrt(128)
fig_nmse_cross, axs_nmse_cross = plt.subplots(2, 2, figsize=(12, 10))
thresh_nmse_cross = 4
hl_alpha_nmse_cross = 0.7

def _nmse_pair(ax, x_col, y_col, x_lbl, y_lbl):
    x_nmse = 1 - summary_df[x_col]
    y_nmse = 1 - summary_df[y_col]
    m = (
        valid_runs_nmse_cross
        & x_nmse.notna()
        & y_nmse.notna()
        & (x_nmse < thresh_nmse_cross)
        & (y_nmse < thresh_nmse_cross)
    )
    ax.scatter(x_nmse[m], y_nmse[m], alpha=0.5)
    if not m.any():
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(x_lbl)
        ax.set_ylabel(y_lbl)
        return
    min_x_idx = x_nmse[m].idxmin()
    min_y_idx = y_nmse[m].idxmin()
    ax.scatter(
        x_nmse.loc[min_x_idx],
        y_nmse.loc[min_x_idx],
        color="magenta",
        s=100,
        label="Min x-axis NMSE",
        edgecolor="k",
        zorder=10,
        alpha=hl_alpha_nmse_cross,
    )
    ax.scatter(
        x_nmse.loc[min_y_idx],
        y_nmse.loc[min_y_idx],
        color="orange",
        s=100,
        label="Min y-axis NMSE",
        edgecolor="k",
        zorder=10,
        alpha=hl_alpha_nmse_cross,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(x_lbl)
    ax.set_ylabel(y_lbl)
    ax.legend()

_nmse_pair(
    axs_nmse_cross[0][0],
    "ms_r2_5",
    "ms_r2_full",
    "NMSE (mean spectrum, first 5)",
    "NMSE (mean spectrum, full)",
)
_nmse_pair(
    axs_nmse_cross[0][1],
    "ts_ms_r2_5",
    "ts_ms_r2_full",
    "NMSE (TS mean spectrum, first 5)",
    "NMSE (TS mean spectrum, full)",
)
_nmse_pair(
    axs_nmse_cross[1][0],
    "ms_r2_5",
    "mean_jac_r2",
    "NMSE (mean spectrum, first 5)",
    "NMSE (mean Jacobian)",
)
_nmse_pair(
    axs_nmse_cross[1][1],
    "ms_r2_full",
    "mean_jac_r2",
    "NMSE (mean spectrum, full)",
    "NMSE (mean Jacobian)",
)

plt.tight_layout()
plt.show()

In [ ]:
summary_df.loc[summary_df['ts_ms_r2_full'].idxmax()]

In [ ]:
summary_df.loc[summary_df['ts_ms_r2_5'].idxmin()]

## Jacobian Consistency Loss Comparison

In [ ]:
# Identify top 5 runs by ms_r2_5 and ms_r2_full
_valid = summary_df.dropna(subset=["ms_r2_5", "ms_r2_full"]).copy()

top5_by_r2_5 = _valid.nlargest(5, "ms_r2_5")["run_id"].tolist()
top5_by_r2_full = _valid.nlargest(5, "ms_r2_full")["run_id"].tolist()

print(
    f"Top 5 by ms_r2_5 (best first-5 exponents): {top5_by_r2_5}\n"
    f"Top 5 by ms_r2_full (best full spectrum): {top5_by_r2_full}\n"
    f"Overlap: {set(top5_by_r2_5) & set(top5_by_r2_full) or 'none'}"
)

In [ ]:
# Load models and compute jac consistency loss on test data
_all_run_ids = list(dict.fromkeys(top5_by_r2_5 + top5_by_r2_full))  # unique, order-preserving
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_data_loaded_jc = False
_trajs_shared_jc = None
_dt_shared_jc = None
_cfg_shared_jc = None

jac_cons_results = {}

for _i, _run_id in enumerate(tqdm(_all_run_ids, desc="Computing jac consistency")):
    try:
        (
            _run_obj, _cfg, _eq, _dt_val, _values,
            _train_dl, _val_dl, _test_dl, _trajs_val, _lit_model,
        ) = load_run(
            PROJECT_PATH,
            run_id=_run_id,
            save_dir=SAVE_DIR,
            generate_data=(not _data_loaded_jc),
            verbose=False,
        )

        if not _data_loaded_jc:
            _trajs_shared_jc = _trajs_val
            _dt_shared_jc = _dt_val
            _cfg_shared_jc = _cfg
            _data_loaded_jc = True

        # Ensure best checkpoint is loaded
        import JacobianODE.jacobians.checkpoints.loader as _ckpt_loader
        try:
            _ckpt_loader.load_checkpoint(
                _run_obj, _cfg, _lit_model,
                save_dir=SAVE_DIR, verbose=False,
            )
        except Exception:
            pass

        _lit_model.eval()
        _lit_model = _lit_model.to(_device)

        _n_target_dims = OmegaConf.select(
            _cfg, "model.n_target_dims", default=None
        )
        if _n_target_dims is not None:
            _n_target_dims = int(_n_target_dims)

        # Test trajectories
        if "test_trajs_full" in _trajs_shared_jc:
            _test_seq = _trajs_shared_jc["test_trajs_full"].sequence
        else:
            _test_seq = _trajs_shared_jc["test_trajs"].sequence

        _traj_t = torch.as_tensor(_test_seq).float().to(_device)

        # Use shared dt (self.dt may be None if not injected during load)
        _dt_run = _lit_model.dt if _lit_model.dt is not None else _dt_shared_jc

        with torch.no_grad():
            _z_full = _lit_model.encode_trajectory(_traj_t)
            _z_dyn = z_dyn_slice(_z_full, _n_target_dims)
            _jacs = _lit_model.compute_jacobians(_z_dyn)

            # Per-trajectory jac consistency loss
            # ||expm(J*dt) @ dz_t - dz_{t+1}||^2 / var(z)
            _losses = []
            for _ti in range(_z_dyn.shape[0]):
                _z_s = _z_dyn[_ti : _ti + 1]   # (1, T, D)
                _j_s = _jacs[_ti : _ti + 1]     # (1, T, D, D)
                _J_exp = torch.matrix_exp(_j_s[:, :-2] * _dt_run)
                _vel = _z_s[:, 1:] - _z_s[:, :-1]
                _vel_pred = (_J_exp @ _vel[:, :-1].unsqueeze(-1)).squeeze(-1)
                _loss = (_vel[:, 1:] - _vel_pred).pow(2).mean() / _z_s.var()
                _losses.append(_loss.item())

            jac_cons_results[_run_id] = {
                "mean": float(np.mean(_losses)),
                "std": float(np.std(_losses)),
                "per_traj": _losses,
            }

        print(
            f"  [{_i+1}/{len(_all_run_ids)}] {_run_id}: "
            f"jac_cons = {jac_cons_results[_run_id]['mean']:.6f} "
            f"+/- {jac_cons_results[_run_id]['std']:.6f}"
        )

        _lit_model.cpu()
        del _lit_model
        torch.cuda.empty_cache()

    except Exception as _e:
        print(f"  Error for {_run_id}: {_e}\n{traceback.format_exc()}")
        jac_cons_results[_run_id] = {
            "mean": float("nan"),
            "std": float("nan"),
            "per_traj": [],
        }

print(f"Computed jac consistency loss for {len(jac_cons_results)} runs.")

In [ ]:
# Compare jac consistency loss: best-first-5 vs best-full-spectrum
from matplotlib.patches import Patch

_df_jc = summary_df.set_index("run_id")
_all_ids = list(dict.fromkeys(top5_by_r2_5 + top5_by_r2_full))

# --- Build comparison table ---
_rows_jc = []
for _rid in _all_ids:
    _r = jac_cons_results.get(_rid, {})
    _in_5 = _rid in top5_by_r2_5
    _in_full = _rid in top5_by_r2_full
    _rows_jc.append({
        "run_id": _rid[:8],
        "group": (
            "both" if (_in_5 and _in_full)
            else "best_r2_5" if _in_5
            else "best_r2_full"
        ),
        "ms_r2_5": _df_jc.loc[_rid, "ms_r2_5"] if _rid in _df_jc.index else None,
        "ms_r2_full": _df_jc.loc[_rid, "ms_r2_full"] if _rid in _df_jc.index else None,
        "jac_cons_mean": _r.get("mean"),
        "jac_cons_std": _r.get("std"),
        "lambda_lc": _df_jc.loc[_rid, "lambda_lc"] if _rid in _df_jc.index else None,
        "kl_dyn": _df_jc.loc[_rid, "kl_dyn"] if _rid in _df_jc.index else None,
        "traj_val_loss": _df_jc.loc[_rid, "traj_val_loss"] if _rid in _df_jc.index else None,
    })
jac_cons_table = pd.DataFrame(_rows_jc).sort_values("group")

# --- Plots ---
fig_jc, (ax_box, ax_scatter) = plt.subplots(1, 2, figsize=(14, 6))

# Left: box plots of per-trajectory jac consistency loss
data_5 = [
    jac_cons_results.get(rid, {}).get("per_traj", []) for rid in top5_by_r2_5
]
data_full = [
    jac_cons_results.get(rid, {}).get("per_traj", []) for rid in top5_by_r2_full
]

pos_5 = list(range(len(top5_by_r2_5)))
pos_full = list(range(
    len(top5_by_r2_5) + 1,
    len(top5_by_r2_5) + 1 + len(top5_by_r2_full),
))

if any(d for d in data_5):
    bp5 = ax_box.boxplot(
        data_5, positions=pos_5, widths=0.6, patch_artist=True
    )
    for patch in bp5["boxes"]:
        patch.set_facecolor("steelblue")
        patch.set_alpha(0.7)
if any(d for d in data_full):
    bpf = ax_box.boxplot(
        data_full, positions=pos_full, widths=0.6, patch_artist=True
    )
    for patch in bpf["boxes"]:
        patch.set_facecolor("coral")
        patch.set_alpha(0.7)

ax_box.set_xticks(pos_5 + pos_full)
ax_box.set_xticklabels(
    [rid[:6] for rid in top5_by_r2_5]
    + [rid[:6] for rid in top5_by_r2_full],
    fontsize=7, rotation=45,
)
ax_box.set_ylabel("Jac Consistency Loss")
ax_box.set_title("Per-Trajectory Jac Consistency Loss")
ax_box.set_yscale("log")
ax_box.legend(handles=[
    Patch(facecolor="steelblue", alpha=0.7, label="Best ms_r2_5 (top-5 exponents)"),
    Patch(facecolor="coral", alpha=0.7, label="Best ms_r2_full (full spectrum)"),
], fontsize=8)

# Right: jac_cons_mean vs full-spectrum NMSE, colored by group
for _rid in _all_ids:
    _r = jac_cons_results.get(_rid, {})
    _jc = _r.get("mean", float("nan"))
    _nmse_full = (
        1 - _df_jc.loc[_rid, "ms_r2_full"]
        if _rid in _df_jc.index else float("nan")
    )
    _in_5 = _rid in top5_by_r2_5
    _in_full = _rid in top5_by_r2_full
    _color = (
        "purple" if (_in_5 and _in_full)
        else "steelblue" if _in_5
        else "coral"
    )
    ax_scatter.scatter(
        _nmse_full, _jc, c=_color, marker="o", s=80,
        alpha=0.8, edgecolors="k", linewidths=0.5,
    )
    ax_scatter.annotate(
        _rid[:6], (_nmse_full, _jc),
        fontsize=6, alpha=0.7,
        xytext=(4, 4), textcoords="offset points",
    )

ax_scatter.set_xlabel("Full Spectrum NMSE (1 - R²)")
ax_scatter.set_ylabel("Jac Consistency Loss (mean)")
ax_scatter.set_title("Jac Consistency vs Full Spectrum NMSE")
ax_scatter.set_xscale("log")
ax_scatter.set_yscale("log")
ax_scatter.legend(handles=[
    Patch(facecolor="steelblue", label="Best ms_r2_5 only"),
    Patch(facecolor="coral", label="Best ms_r2_full only"),
    Patch(facecolor="purple", label="Both"),
], fontsize=8)

plt.tight_layout()

# Group-level summary
_g5_means = [
    jac_cons_results.get(rid, {}).get("mean", float("nan"))
    for rid in top5_by_r2_5
]
_gf_means = [
    jac_cons_results.get(rid, {}).get("mean", float("nan"))
    for rid in top5_by_r2_full
]

print(
    f"Group means:  "
    f"Best-r2-5 = {np.nanmean(_g5_means):.6f},  "
    f"Best-r2-full = {np.nanmean(_gf_means):.6f}  "
    f"(ratio = {np.nanmean(_g5_means) / np.nanmean(_gf_means):.2f}x)"
)

plt.show()
jac_cons_table

## Jacobian Consistency Loss Comparison

In [ ]:
# Identify top 5 runs by ms_r2_5 and ms_r2_full
_valid = summary_df.dropna(subset=["ms_r2_5", "ms_r2_full"]).copy()

top5_by_r2_5 = _valid.nlargest(5, "ms_r2_5")["run_id"].tolist()
top5_by_r2_full = _valid.nlargest(5, "ms_r2_full")["run_id"].tolist()

print(
    f"Top 5 by ms_r2_5 (best first-5 exponents): {top5_by_r2_5}\n"
    f"Top 5 by ms_r2_full (best full spectrum): {top5_by_r2_full}\n"
    f"Overlap: {set(top5_by_r2_5) & set(top5_by_r2_full) or 'none'}"
)

In [ ]:
# Load models and compute jac consistency loss on test data
_all_run_ids = list(dict.fromkeys(top5_by_r2_5 + top5_by_r2_full))  # unique, order-preserving
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_data_loaded_jc = False
_trajs_shared_jc = None
_dt_shared_jc = None
_cfg_shared_jc = None

jac_cons_results = {}

for _i, _run_id in enumerate(tqdm(_all_run_ids, desc="Computing jac consistency")):
    try:
        (
            _run_obj, _cfg, _eq, _dt_val, _values,
            _train_dl, _val_dl, _test_dl, _trajs_val, _lit_model,
        ) = load_run(
            PROJECT_PATH,
            run_id=_run_id,
            save_dir=SAVE_DIR,
            generate_data=(not _data_loaded_jc),
            verbose=False,
        )

        if not _data_loaded_jc:
            _trajs_shared_jc = _trajs_val
            _dt_shared_jc = _dt_val
            _cfg_shared_jc = _cfg
            _data_loaded_jc = True

        # Ensure best checkpoint is loaded
        import JacobianODE.jacobians.checkpoints.loader as _ckpt_loader
        try:
            _ckpt_loader.load_checkpoint(
                _run_obj, _cfg, _lit_model,
                save_dir=SAVE_DIR, verbose=False,
            )
        except Exception:
            pass

        _lit_model.eval()
        _lit_model = _lit_model.to(_device)

        _n_target_dims = OmegaConf.select(
            _cfg, "model.n_target_dims", default=None
        )
        if _n_target_dims is not None:
            _n_target_dims = int(_n_target_dims)

        # Test trajectories
        if "test_trajs_full" in _trajs_shared_jc:
            _test_seq = _trajs_shared_jc["test_trajs_full"].sequence
        else:
            _test_seq = _trajs_shared_jc["test_trajs"].sequence

        _traj_t = torch.as_tensor(_test_seq).float().to(_device)

        # Use shared dt (self.dt may be None if not injected during load)
        _dt_run = _lit_model.dt if _lit_model.dt is not None else _dt_shared_jc

        with torch.no_grad():
            _z_full = _lit_model.encode_trajectory(_traj_t)
            _z_dyn = z_dyn_slice(_z_full, _n_target_dims)
            _jacs = _lit_model.compute_jacobians(_z_dyn)

            # Per-trajectory jac consistency loss (inline to avoid self.dt=None)
            # ||expm(J*dt) @ dz_t - dz_{t+1}||^2 / var(z)
            _losses = []
            for _ti in range(_z_dyn.shape[0]):
                _z_s = _z_dyn[_ti : _ti + 1]   # (1, T, D)
                _j_s = _jacs[_ti : _ti + 1]     # (1, T, D, D)
                _J_exp = torch.matrix_exp(_j_s[:, :-2] * _dt_run)
                _vel = _z_s[:, 1:] - _z_s[:, :-1]
                _vel_pred = (_J_exp @ _vel[:, :-1].unsqueeze(-1)).squeeze(-1)
                _loss = (_vel[:, 1:] - _vel_pred).pow(2).mean() / _z_s.var()
                _losses.append(_loss.item())

            jac_cons_results[_run_id] = {
                "mean": float(np.mean(_losses)),
                "std": float(np.std(_losses)),
                "per_traj": _losses,
            }

        print(
            f"  [{_i+1}/{len(_all_run_ids)}] {_run_id}: "
            f"jac_cons = {jac_cons_results[_run_id]['mean']:.6f} "
            f"+/- {jac_cons_results[_run_id]['std']:.6f}"
        )

        _lit_model.cpu()
        del _lit_model
        torch.cuda.empty_cache()

    except Exception as _e:
        print(f"  Error for {_run_id}: {_e}\n{traceback.format_exc()}")
        jac_cons_results[_run_id] = {
            "mean": float("nan"),
            "std": float("nan"),
            "per_traj": [],
        }

print(f"Computed jac consistency loss for {len(jac_cons_results)} runs.")

In [ ]:
# Compare jac consistency loss: best-first-5 vs best-full-spectrum
from matplotlib.patches import Patch

_df_jc = summary_df.set_index("run_id")
_all_ids = list(dict.fromkeys(top5_by_r2_5 + top5_by_r2_full))

# --- Build comparison table ---
_rows_jc = []
for _rid in _all_ids:
    _r = jac_cons_results.get(_rid, {})
    _in_5 = _rid in top5_by_r2_5
    _in_full = _rid in top5_by_r2_full
    _rows_jc.append({
        "run_id": _rid[:8],
        "group": (
            "both" if (_in_5 and _in_full)
            else "best_r2_5" if _in_5
            else "best_r2_full"
        ),
        "ms_r2_5": _df_jc.loc[_rid, "ms_r2_5"] if _rid in _df_jc.index else None,
        "ms_r2_full": _df_jc.loc[_rid, "ms_r2_full"] if _rid in _df_jc.index else None,
        "jac_cons_mean": _r.get("mean"),
        "jac_cons_std": _r.get("std"),
        "lambda_lc": _df_jc.loc[_rid, "lambda_lc"] if _rid in _df_jc.index else None,
        "kl_dyn": _df_jc.loc[_rid, "kl_dyn"] if _rid in _df_jc.index else None,
        "traj_val_loss": _df_jc.loc[_rid, "traj_val_loss"] if _rid in _df_jc.index else None,
    })
jac_cons_table = pd.DataFrame(_rows_jc).sort_values("group")

# --- Plots ---
fig_jc, (ax_box, ax_scatter) = plt.subplots(1, 2, figsize=(14, 6))

# Left: box plots of per-trajectory jac consistency loss
data_5 = [
    jac_cons_results.get(rid, {}).get("per_traj", []) for rid in top5_by_r2_5
]
data_full = [
    jac_cons_results.get(rid, {}).get("per_traj", []) for rid in top5_by_r2_full
]

pos_5 = list(range(len(top5_by_r2_5)))
pos_full = list(range(
    len(top5_by_r2_5) + 1,
    len(top5_by_r2_5) + 1 + len(top5_by_r2_full),
))

if any(d for d in data_5):
    bp5 = ax_box.boxplot(
        data_5, positions=pos_5, widths=0.6, patch_artist=True
    )
    for patch in bp5["boxes"]:
        patch.set_facecolor("steelblue")
        patch.set_alpha(0.7)
if any(d for d in data_full):
    bpf = ax_box.boxplot(
        data_full, positions=pos_full, widths=0.6, patch_artist=True
    )
    for patch in bpf["boxes"]:
        patch.set_facecolor("coral")
        patch.set_alpha(0.7)

ax_box.set_xticks(pos_5 + pos_full)
ax_box.set_xticklabels(
    [rid[:6] for rid in top5_by_r2_5]
    + [rid[:6] for rid in top5_by_r2_full],
    fontsize=7, rotation=45,
)
ax_box.set_ylabel("Jac Consistency Loss")
ax_box.set_title("Per-Trajectory Jac Consistency Loss")
ax_box.set_yscale("log")
ax_box.legend(handles=[
    Patch(facecolor="steelblue", alpha=0.7, label="Best ms_r2_5 (top-5 exponents)"),
    Patch(facecolor="coral", alpha=0.7, label="Best ms_r2_full (full spectrum)"),
], fontsize=8)

# Right: jac_cons_mean vs full-spectrum NMSE, colored by group
for _rid in _all_ids:
    _r = jac_cons_results.get(_rid, {})
    _jc = _r.get("mean", float("nan"))
    _nmse_full = (
        1 - _df_jc.loc[_rid, "ms_r2_full"]
        if _rid in _df_jc.index else float("nan")
    )
    _in_5 = _rid in top5_by_r2_5
    _in_full = _rid in top5_by_r2_full
    _color = (
        "purple" if (_in_5 and _in_full)
        else "steelblue" if _in_5
        else "coral"
    )
    ax_scatter.scatter(
        _nmse_full, _jc, c=_color, marker="o", s=80,
        alpha=0.8, edgecolors="k", linewidths=0.5,
    )
    ax_scatter.annotate(
        _rid[:6], (_nmse_full, _jc),
        fontsize=6, alpha=0.7,
        xytext=(4, 4), textcoords="offset points",
    )

ax_scatter.set_xlabel("Full Spectrum NMSE (1 - R²)")
ax_scatter.set_ylabel("Jac Consistency Loss (mean)")
ax_scatter.set_title("Jac Consistency vs Full Spectrum NMSE")
ax_scatter.set_xscale("log")
ax_scatter.set_yscale("log")
ax_scatter.legend(handles=[
    Patch(facecolor="steelblue", label="Best ms_r2_5 only"),
    Patch(facecolor="coral", label="Best ms_r2_full only"),
    Patch(facecolor="purple", label="Both"),
], fontsize=8)

plt.tight_layout()
plt.show()

# Group-level summary
_g5_means = [
    jac_cons_results.get(rid, {}).get("mean", float("nan"))
    for rid in top5_by_r2_5
]
_gf_means = [
    jac_cons_results.get(rid, {}).get("mean", float("nan"))
    for rid in top5_by_r2_full
]

print(
    f"Group means:  "
    f"Best-r2-5 = {np.nanmean(_g5_means):.6f},  "
    f"Best-r2-full = {np.nanmean(_gf_means):.6f}  "
    f"(ratio = {np.nanmean(_g5_means) / np.nanmean(_gf_means):.2f}x)"
)
jac_cons_table

## HTML Report Generation

In [ ]:
_CSS = """
body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
       max-width: 1200px; margin: 0 auto; padding: 24px 32px; background: #fff; color: #222; }
h1   { font-size: 1.6em; margin-bottom: 0.15em; }
h2   { font-size: 1.15em; color: #444; border-bottom: 2px solid #e0e0e0;
        padding-bottom: 4px; margin-top: 2.2em; }
pre  { background: #f6f8fa; border: 1px solid #e1e4e8; border-radius: 6px;
        padding: 12px 16px; font-size: 12.5px; line-height: 1.6;
        overflow-x: auto; white-space: pre-wrap; word-break: break-word; }
.figure { margin: 18px 0; text-align: center; }
img  { max-width: 100%; border: 1px solid #e0e0e0; border-radius: 4px; }
.subtitle { color: #666; font-size: 0.9em; margin-bottom: 2em; }
table { border-collapse: collapse; margin: 12px 0; }
th, td { border: 1px solid #e0e0e0; padding: 6px 12px; text-align: left; font-size: 13px; }
th { background: #f6f8fa; }
"""

def make_html_report(title, subtitle, sections):
    """Build a self-contained HTML report.

    sections: list of (heading, content_html) tuples
    """
    body_parts = []
    for heading, content in sections:
        body_parts.append(f"<h2>{heading}</h2>\n{content}")
    body = "\n".join(body_parts)
    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{title}</title>
  <style>{_CSS}</style>
</head>
<body>
<h1>{title}</h1>
<p class="subtitle">{subtitle}</p>
{body}
</body>
</html>"""

def fig_html(fig_or_b64):
    if isinstance(fig_or_b64, str):
        b64 = fig_or_b64
    else:
        b64 = fig_to_base64(fig_or_b64)
    return f'<div class="figure"><img src="data:image/png;base64,{b64}"/></div>'

def table_html(df):
    return df.to_html(index=False, na_rep="--", float_format=lambda x: f"{x:.4f}")

# --- Per-run HTML reports ---
_repo_root = Path.cwd().parent if Path.cwd().name == "_notebook" else Path.cwd()
_per_run_dir = results_dir
for _run_id in discovered.run_ids:
    _r = all_results.get(_run_id, {})
    if "error" in _r:
        continue
    _cfg = _r.get("config", {})
    _sections = []

    # Config section
    _config_lines = "\n".join(f"{k}: {v}" for k, v in _cfg.items())
    _sections.append(("Configuration", f"<pre>{_config_lines}</pre>"))

    # Lyapunov spectrum
    _lyap_lines = []
    _pred_mean = _r.get("pred_lyap_mean", [])
    _pred_std = _r.get("pred_lyap_std", [])
    _emp_mean = _r.get("emp_lyap_mean", [])
    _emp_std = _r.get("emp_lyap_std", [])
    _lyap_lines.append("Predicted (mean +/- std):")
    for _i, (_le, _sd) in enumerate(zip(_pred_mean, _pred_std)):
        _lyap_lines.append(f"  LE_{_i+1} = {_le:+.4f} +/- {_sd:.4f}")
    _lyap_lines.append("Empirical (mean +/- std):")
    for _i, (_le, _sd) in enumerate(zip(_emp_mean, _emp_std)):
        _lyap_lines.append(f"  LE_{_i+1} = {_le:+.4f} +/- {_sd:.4f}")
    _lyap_lines.append(f"Mean R^2:   {_r.get('mean_spectrum_r2', float('nan')):.4f}")
    _lyap_lines.append(f"Mean corr:  {_r.get('mean_spectrum_corr', float('nan')):.4f}")
    _content = "<pre>" + "\n".join(_lyap_lines) + "</pre>"
    if _run_id in lyap_figs:
        _content += fig_html(fig_to_base64(lyap_figs[_run_id]))
    _sections.append(("Lyapunov Spectrum", _content))

    # Obs-space Jacobian (if computed)
    if _r.get("obs_jac_lyap_mean"):
        _obs_lines = ["Observed-space Jacobian Lyapunov (mean +/- std):"]
        for _i, (_le, _sd) in enumerate(
            zip(_r["obs_jac_lyap_mean"], _r.get("obs_jac_lyap_std", []))
        ):
            _obs_lines.append(f"  LE_{_i+1} = {_le:+.4f} +/- {_sd:.4f}")
        if "mean_jac_r2" in _r:
            _obs_lines.append(f"Mean Jacobian R^2: {_r['mean_jac_r2']:.4f}")
        _sections.append(
            ("Observed-Space Jacobian", "<pre>" + "\n".join(_obs_lines) + "</pre>")
        )

    _html = make_html_report(
        f"Lyapunov Report - {_run_id}",
        f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
        f"Project: {PROJECT} | Group: {GROUP or '(none)'}",
        _sections,
    )
    (_per_run_dir / f"{_run_id}.html").write_text(_html, encoding="utf-8")

# --- Aggregate HTML report ---
_report_dir = _repo_root / "_marimo" / "reports"
_report_dir.mkdir(parents=True, exist_ok=True)
_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
_agg_path = _report_dir / f"{_timestamp} - Lyapunov_{PROJECT}_{GROUP or 'all'}.html"

_agg_sections = []

# Summary table (sorted by full mean-spectrum R² descending)
_sorted_df = summary_df.sort_values("ms_r2_full", ascending=False, na_position="last")
_agg_sections.append(("Summary", table_html(_sorted_df)))

# Cross-run plot
if cross_fig is not None:
    _agg_sections.append(("Cross-Run Comparison", fig_html(fig_to_base64(cross_fig))))

# Scatter plots (all generated grids, in sorted key order)
for _key in sorted(scatter_figs.keys()):
    _fig = scatter_figs[_key]
    if _fig is not None:
        _agg_sections.append((_key.replace("_", " ").title(), fig_html(fig_to_base64(_fig))))

# Per-run spectra
for _run_id in discovered.run_ids:
    if _run_id in lyap_figs:
        _r = all_results.get(_run_id, {})
        _r2 = _r.get("mean_spectrum_r2", float("nan"))
        _agg_sections.append(
            (
                f"Run {_run_id} (R^2={_r2:.4f})",
                fig_html(fig_to_base64(lyap_figs[_run_id])),
            )
        )

_agg_html = make_html_report(
    f"Lyapunov Spectrum Analysis - {PROJECT}",
    f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
    f"Entity: {ENTITY} | Project: {PROJECT} | Group: {GROUP or '(none)'} | "
    f"Runs: {len(discovered.run_ids)}",
    _agg_sections,
)
_agg_path.write_text(_agg_html, encoding="utf-8")

print(
    f"Reports saved.\n"
    f"- Per-run HTML: {_per_run_dir}/\n"
    f"- Aggregate: {_agg_path}"
)

## Exploratory Analysis

In [ ]:
summary_df.columns

In [ ]:
summary_df.loc[summary_df['traj_val_loss'].argsort()]

In [ ]:
valid_runs = summary_df['val_lc_loss'] <= np.sqrt(0.5)
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

thresh = 3
# ms_r2_5
valid_runs_ax0 = valid_runs & ((1 - summary_df['ms_r2_5']) < thresh)
axs[0][0].scatter(summary_df['traj_val_loss'][valid_runs_ax0], 1 - summary_df['ms_r2_5'][valid_runs_ax0])
min_traj_val_idx_0 = summary_df['traj_val_loss'][valid_runs_ax0].idxmin()
axs[0][0].scatter(
    summary_df.loc[min_traj_val_idx_0, 'traj_val_loss'], 
    1 - summary_df.loc[min_traj_val_idx_0, 'ms_r2_5'],
    color='magenta', s=100, label='Min Traj Val Loss', edgecolor='k', zorder=10, alpha=0.7
)
min_nmse_idx_0 = (1 - summary_df['ms_r2_5'][valid_runs_ax0]).idxmin()
axs[0][0].scatter(
    summary_df.loc[min_nmse_idx_0, 'traj_val_loss'], 
    1 - summary_df.loc[min_nmse_idx_0, 'ms_r2_5'],
    color='orange', s=100, label='Min NMSE', edgecolor='k', zorder=10, alpha=0.7
)
axs[0][0].set_xscale('log')
axs[0][0].set_yscale('log')
axs[0][0].set_xlabel('Trajectory Validation Loss')
axs[0][0].set_ylabel('Mean Spectrum\nNormalized MSE\n(first 5 exponents)')
axs[0][0].legend()

# ms_r2_full
valid_runs_ax1 = valid_runs & ((1 - summary_df['ms_r2_full']) < thresh)
axs[0][1].scatter(summary_df['traj_val_loss'][valid_runs_ax1], 1 - summary_df['ms_r2_full'][valid_runs_ax1])
min_traj_val_idx_1 = summary_df['traj_val_loss'][valid_runs_ax1].idxmin()
axs[0][1].scatter(
    summary_df.loc[min_traj_val_idx_1, 'traj_val_loss'], 
    1 - summary_df.loc[min_traj_val_idx_1, 'ms_r2_full'], 
    color='magenta', s=100, label='Min Traj Val Loss', edgecolor='k', zorder=10, alpha=0.7
)
min_nmse_idx_1 = (1 - summary_df['ms_r2_full'][valid_runs_ax1]).idxmin()
axs[0][1].scatter(
    summary_df.loc[min_nmse_idx_1, 'traj_val_loss'], 
    1 - summary_df.loc[min_nmse_idx_1, 'ms_r2_full'],
    color='orange', s=100, label='Min NMSE', edgecolor='k', zorder=10, alpha=0.7
)
axs[0][1].set_xscale('log')
axs[0][1].set_yscale('log')
axs[0][1].set_xlabel('Trajectory Validation Loss')
axs[0][1].set_ylabel('Mean Spectrum\nNormalized MSE\n(full spectrum)')
axs[0][1].legend()

# ts_ms_r2_5
valid_runs_ax2 = valid_runs & ((1 - summary_df['ts_ms_r2_5']) < thresh)
axs[1][0].scatter(summary_df['traj_val_loss'][valid_runs_ax2], 1 - summary_df['ts_ms_r2_5'][valid_runs_ax2])
min_traj_val_idx_2 = summary_df['traj_val_loss'][valid_runs_ax2].idxmin()
axs[1][0].scatter(
    summary_df.loc[min_traj_val_idx_2, 'traj_val_loss'], 
    1 - summary_df.loc[min_traj_val_idx_2, 'ts_ms_r2_5'], 
    color='magenta', s=100, label='Min Traj Val Loss', edgecolor='k', zorder=10, alpha=0.7
)
min_nmse_idx_2 = (1 - summary_df['ts_ms_r2_5'][valid_runs_ax2]).idxmin()
axs[1][0].scatter(
    summary_df.loc[min_nmse_idx_2, 'traj_val_loss'], 
    1 - summary_df.loc[min_nmse_idx_2, 'ts_ms_r2_5'],
    color='orange', s=100, label='Min NMSE', edgecolor='k', zorder=10, alpha=0.7
)
axs[1][0].set_xscale('log')
axs[1][0].set_yscale('log')
axs[1][0].set_xlabel('Trajectory Validation Loss')
axs[1][0].set_ylabel('Time Series Mean Spectrum\nNormalized MSE\n(first 5 exponents)')
axs[1][0].legend()

# ts_ms_r2_full
valid_runs_ax3 = valid_runs & ((1 - summary_df['ts_ms_r2_full']) < thresh)
axs[1][1].scatter(summary_df['traj_val_loss'][valid_runs_ax3], 1 - summary_df['ts_ms_r2_full'][valid_runs_ax3])
min_traj_val_idx_3 = summary_df['traj_val_loss'][valid_runs_ax3].idxmin()
axs[1][1].scatter(
    summary_df.loc[min_traj_val_idx_3, 'traj_val_loss'], 
    1 - summary_df.loc[min_traj_val_idx_3, 'ts_ms_r2_full'], 
    color='magenta', s=100, label='Min Traj Val Loss', edgecolor='k', zorder=10, alpha=0.7
)
min_nmse_idx_3 = (1 - summary_df['ts_ms_r2_full'][valid_runs_ax3]).idxmin()
axs[1][1].scatter(
    summary_df.loc[min_nmse_idx_3, 'traj_val_loss'], 
    1 - summary_df.loc[min_nmse_idx_3, 'ts_ms_r2_full'],
    color='orange', s=100, label='Min NMSE', edgecolor='k', zorder=10, alpha=0.7
)
axs[1][1].set_xscale('log')
axs[1][1].set_yscale('log')
axs[1][1].set_xlabel('Trajectory Validation Loss')
axs[1][1].set_ylabel('Time Series Mean Spectrum\nNormalized MSE\n(full spectrum)')
axs[1][1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cross-metric NMSE scatter (log-log): magenta = argmin x NMSE, orange = argmin y NMSE
valid_runs_nmse_cross = summary_df["val_lc_loss"] <= np.sqrt(128)
fig_nmse_cross, axs_nmse_cross = plt.subplots(2, 2, figsize=(12, 10))
thresh_nmse_cross = 4
hl_alpha_nmse_cross = 0.7

def _nmse_pair(ax, x_col, y_col, x_lbl, y_lbl):
    x_nmse = 1 - summary_df[x_col]
    y_nmse = 1 - summary_df[y_col]
    m = (
        valid_runs_nmse_cross
        & x_nmse.notna()
        & y_nmse.notna()
        & (x_nmse < thresh_nmse_cross)
        & (y_nmse < thresh_nmse_cross)
    )
    ax.scatter(x_nmse[m], y_nmse[m], alpha=0.5)
    if not m.any():
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(x_lbl)
        ax.set_ylabel(y_lbl)
        return
    min_x_idx = x_nmse[m].idxmin()
    min_y_idx = y_nmse[m].idxmin()
    ax.scatter(
        x_nmse.loc[min_x_idx],
        y_nmse.loc[min_x_idx],
        color="magenta",
        s=100,
        label="Min x-axis NMSE",
        edgecolor="k",
        zorder=10,
        alpha=hl_alpha_nmse_cross,
    )
    ax.scatter(
        x_nmse.loc[min_y_idx],
        y_nmse.loc[min_y_idx],
        color="orange",
        s=100,
        label="Min y-axis NMSE",
        edgecolor="k",
        zorder=10,
        alpha=hl_alpha_nmse_cross,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(x_lbl)
    ax.set_ylabel(y_lbl)
    ax.legend()

_nmse_pair(
    axs_nmse_cross[0][0],
    "ms_r2_5",
    "ms_r2_full",
    "NMSE (mean spectrum, first 5)",
    "NMSE (mean spectrum, full)",
)
_nmse_pair(
    axs_nmse_cross[0][1],
    "ts_ms_r2_5",
    "ts_ms_r2_full",
    "NMSE (TS mean spectrum, first 5)",
    "NMSE (TS mean spectrum, full)",
)
_nmse_pair(
    axs_nmse_cross[1][0],
    "ms_r2_5",
    "mean_jac_r2",
    "NMSE (mean spectrum, first 5)",
    "NMSE (mean Jacobian)",
)
_nmse_pair(
    axs_nmse_cross[1][1],
    "ms_r2_full",
    "mean_jac_r2",
    "NMSE (mean spectrum, full)",
    "NMSE (mean Jacobian)",
)

plt.tight_layout()
plt.show()

In [ ]:
summary_df.loc[summary_df['ts_ms_r2_full'].idxmax()]

In [ ]:
summary_df.loc[summary_df['ts_ms_r2_5'].idxmin()]

## Per-Run Lyapunov Spectrum Plots

In [ ]:
lyap_figs = {}
for _run_id in discovered.run_ids:
    _r = all_results.get(_run_id, {})
    if "error" in _r or "pred_lyap_mean" not in _r:
        continue

    _pred_mean = np.array(_r["pred_lyap_mean"])
    _pred_std = np.array(_r["pred_lyap_std"])
    _emp_mean = np.array(_r["emp_lyap_mean"])
    _emp_std = np.array(_r["emp_lyap_std"])
    _lc = _r.get("config", {}).get("loop_closure_weight")

    _result = plot_lyapunov_spectrum(
        _pred_mean,
        _pred_std,
        _emp_mean,
        _emp_std,
        true_lyapunov=TRUE_LYAPUNOV,
        loop_closure_weight=_lc,
    )
    if isinstance(_result, list):
        _fig = _result[0]
    else:
        _fig = _result
    _fig.suptitle(f"Run {_run_id} (lc={_lc})", fontsize=10, y=1.02)
    lyap_figs[_run_id] = _fig

print(f"{len(lyap_figs)} Lyapunov spectrum plots generated.")

## Cross-Run Comparison Plots

In [ ]:
_lcs = []
_r2s = []
_pred_le1s = []
_emp_le1s = []
_labels = []

for _run_id in discovered.run_ids:
    _r = all_results.get(_run_id, {})
    if "error" in _r or "mean_spectrum_r2" not in _r:
        continue
    _cfg = _r.get("config", {})
    _lcs.append(_cfg.get("loop_closure_weight", 0))
    _r2s.append(_r["mean_spectrum_r2"])
    _pred = _r.get("pred_lyap_mean", [])
    _emp = _r.get("emp_lyap_mean", [])
    _pred_le1s.append(_pred[0] if _pred else float("nan"))
    _emp_le1s.append(_emp[0] if _emp else float("nan"))
    _labels.append(_run_id[:8])

if _lcs:
    cross_fig, (_ax1, _ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Largest LE comparison
    _ax1.scatter(_emp_le1s, _pred_le1s, c=np.log10(np.array(_lcs) + 1e-10), cmap="viridis", s=60)
    _lim = [
        min(min(_emp_le1s), min(_pred_le1s)) - 0.1,
        max(max(_emp_le1s), max(_pred_le1s)) + 0.1,
    ]
    _ax1.plot(_lim, _lim, "k--", alpha=0.3, label="y=x")
    _ax1.set_xlabel("Empirical LE_1")
    _ax1.set_ylabel("Predicted LE_1")
    _ax1.set_title("Largest Lyapunov Exponent")
    _ax1.legend()

    # Right: R^2 vs loop_closure_weight
    _ax2.scatter(_lcs, _r2s, s=60, c="steelblue")
    _ax2.set_xscale("symlog", linthresh=1e-7)
    _ax2.set_xlabel("loop_closure_weight")
    _ax2.set_ylabel("Mean Spectrum R^2")
    _ax2.set_title("Spectrum R^2 vs Loop Closure Weight")
    _ax2.axhline(y=1.0, color="k", linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    cross_fig = None
    print("No runs to compare.")

## Scatter Plots: R²/MSE Metrics vs Val Losses

In [ ]:
_df = summary_df.dropna(subset=["lambda_lc"]).copy()

# Add n_target_dims from cached results config
_df["n_target_dims"] = _df["run_id"].map(
    lambda rid: all_results.get(rid, {}).get("config", {}).get("n_target_dims")
)

def collect_available(col_label_pairs, df):
    """Filter to columns that exist and have data."""
    return [(c, l) for c, l in col_label_pairs if c in df.columns and df[c].notna().any()]

# -- Exponent-space R² columns --
_exp_r2_cols = collect_available([
    ("pt_r2_5", "Per-Traj R² (top 5)"),
    ("pt_r2_10", "Per-Traj R² (top 10)"),
    ("pt_r2_full", "Per-Traj R² (full)"),
    ("ms_r2_5", "Mean-Spec R² (top 5)"),
    ("ms_r2_10", "Mean-Spec R² (top 10)"),
    ("ms_r2_full", "Mean-Spec R² (full)"),
    ("mean_jac_r2", "Jacobian R²"),
], _df)

# -- Exponent-space MSE columns --
_exp_mse_cols = collect_available([
    ("pt_mse_5", "Per-Traj MSE (top 5)"),
    ("pt_mse_10", "Per-Traj MSE (top 10)"),
    ("pt_mse_full", "Per-Traj MSE (full)"),
    ("ms_mse_5", "Mean-Spec MSE (top 5)"),
    ("ms_mse_10", "Mean-Spec MSE (top 10)"),
    ("ms_mse_full", "Mean-Spec MSE (full)"),
], _df)

# -- Timescale-space R² columns --
_ts_r2_cols = collect_available([
    ("ts_pt_r2_5", "TS Per-Traj R² (top 5)"),
    ("ts_pt_r2_10", "TS Per-Traj R² (top 10)"),
    ("ts_pt_r2_full", "TS Per-Traj R² (full)"),
    ("ts_ms_r2_5", "TS Mean-Spec R² (top 5)"),
    ("ts_ms_r2_10", "TS Mean-Spec R² (top 10)"),
    ("ts_ms_r2_full", "TS Mean-Spec R² (full)"),
], _df)

# -- Timescale-space MSE columns --
_ts_mse_cols = collect_available([
    ("ts_pt_mse_5", "TS Per-Traj MSE (top 5)"),
    ("ts_pt_mse_10", "TS Per-Traj MSE (top 10)"),
    ("ts_pt_mse_full", "TS Per-Traj MSE (full)"),
    ("ts_ms_mse_5", "TS Mean-Spec MSE (top 5)"),
    ("ts_ms_mse_10", "TS Mean-Spec MSE (top 10)"),
    ("ts_ms_mse_full", "TS Mean-Spec MSE (full)"),
], _df)

def make_scatter_grid(df, x_col, x_label, title_prefix, metric_cols,
                      y_label="R²", y_clip=(-15, 1.1), log_x=False, log_y=False):
    """Scatter subplots: each metric vs x_col."""
    _n = len(metric_cols)
    if _n == 0:
        return None
    _ncols = min(4, _n)
    _nrows = (_n + _ncols - 1) // _ncols
    fig, axes = plt.subplots(_nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows), squeeze=False)

    for _idx, (_col, _label) in enumerate(metric_cols):
        _ax = axes[_idx // _ncols][_idx % _ncols]
        _mask = df[_col].notna() & df[x_col].notna()
        _x = df.loc[_mask, x_col].values
        _y = df.loc[_mask, _col].values
        if not log_y:
            _y = np.clip(_y, y_clip[0], y_clip[1])

        _ax.scatter(_x, _y, s=25, alpha=0.7, c="steelblue", edgecolors="k", linewidths=0.3)
        if not log_y:
            _ax.axhline(y=1.0, color="k", linestyle="--", alpha=0.3, linewidth=0.8)
            _ax.axhline(y=0.0, color="gray", linestyle=":", alpha=0.3, linewidth=0.8)
            _ax.set_ylim(y_clip)
        if log_x:
            _ax.set_xscale("log")
        if log_y:
            _ax.set_yscale("log")
        _ax.set_xlabel(x_label, fontsize=9)
        _ax.set_ylabel(y_label, fontsize=9)
        _ax.set_title(_label, fontsize=10)

    for _idx2 in range(_n, _nrows * _ncols):
        axes[_idx2 // _ncols][_idx2 % _ncols].set_visible(False)

    fig.suptitle(title_prefix, fontsize=13, y=1.02)
    plt.tight_layout()
    return fig

scatter_figs = {}

if _df.empty:
    print("No data for scatter plots.")
else:
    # Build list of (metric_group_name, cols, y_label, y_clip, log_y)
    _metric_groups = [
        ("exp_r2", _exp_r2_cols, "Exponent R²", "R²", (-15, 1.1), False),
        ("exp_mse", _exp_mse_cols, "Exponent MSE", "MSE", None, True),
        ("ts_r2", _ts_r2_cols, "Timescale R²", "R²", (-15, 1.1), False),
        ("ts_mse", _ts_mse_cols, "Timescale MSE", "MSE", None, True),
    ]

    # 3 x-axis groups
    _x_groups = []

    # Group 1: all runs vs traj val loss
    _df1 = _df.dropna(subset=["traj_val_loss"])
    if not _df1.empty:
        _x_groups.append(("traj_vloss", _df1, "traj_val_loss", "trajectory val loss", "all runs"))

    # Group 2: all runs vs LC val loss
    _df2 = _df.dropna(subset=["val_lc_loss"])
    if not _df2.empty:
        _x_groups.append(("lc_vloss", _df2, "val_lc_loss", "loop closure val loss", "all runs"))

    # Group 3: feasible runs vs traj val loss
    _df3 = _df.dropna(subset=["traj_val_loss", "val_lc_loss", "n_target_dims"]).copy()
    if not _df3.empty:
        _df3 = _df3[_df3["val_lc_loss"] <= np.sqrt(_df3["n_target_dims"])]
        if not _df3.empty:
            _x_groups.append(("traj_vloss_feasible", _df3, "traj_val_loss", "trajectory val loss",
                              f"feasible (n={len(_df3)})"))

    for _xkey, _xdf, _xcol, _xlabel, _xsuffix in _x_groups:
        for _mkey, _mcols, _mtitle, _ylabel, _yclip, _logy in _metric_groups:
            if not _mcols:
                continue
            _fig = make_scatter_grid(
                _xdf, _xcol, _xlabel,
                f"{_mtitle} vs {_xlabel} ({_xsuffix})",
                _mcols, y_label=_ylabel, y_clip=_yclip, log_x=True, log_y=_logy,
            )
            if _fig is not None:
                scatter_figs[f"{_mkey}_vs_{_xkey}"] = _fig

    print(f"Generated {len(scatter_figs)} scatter plot grids.")
    plt.show()